# 10 — Community detection sulle reti di keyword e di autori

Cerchiamo gruppi di nodi più collegati tra loro che con il resto della rete (comunità), su due tipi di rete:

- **reti di keyword** (co-occorrenza), NYT (notebook 02) e Guardian (notebook 08): una comunità è un gruppo di temi che la redazione associa negli stessi articoli;
- **rete degli autori NYT** (bipartito autore-keyword del notebook 03): una comunità è un gruppo di giornalisti che scrivono degli stessi temi.

**Input** (letti da disco, nessuna rete ricostruita dagli articoli):

- `data/processed/nyt/ice_topic_network.gexf`, `ice_topic_network_pruned.gexf` (+ grafi per periodo)
- `data/processed/guardian/ice_topic_network.gexf`, `ice_topic_network_pruned.gexf` (+ grafi per periodo)
- `data/processed/nyt/ice_bipartite_author_keyword.gexf`
- `data/processed/nyt/articles_ice.parquet`, `data/processed/guardian/articles_ice_relevant.parquet` (solo per il numero di articoli)

**Struttura**:

0. Setup
1. Caricamento delle reti
2. Pesi delle reti di keyword: association strength
3. Rete degli autori: similarità e backbone
4. Algoritmo (Leiden), risoluzione e stabilità
5. Comunità delle keyword (NYT e Guardian)
6. Comunità degli autori
7. Comunità per periodo (non svolta, vedi la nota in §7)
8. Export e note per il report

Descrizione dei metodi in `docs/community_detection.md`.

## 0. Setup

In [ ]:
import html
import json
import re
import logging
from collections import Counter
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NYT_DIR = BASE_DIR / "data" / "processed" / "nyt"
GUARDIAN_DIR = BASE_DIR / "data" / "processed" / "guardian"

PAPERS = {
    "nyt": {"dir": NYT_DIR, "articles": NYT_DIR / "articles_ice.parquet"},
    "guardian": {"dir": GUARDIAN_DIR, "articles": GUARDIAN_DIR / "articles_ice_relevant.parquet"},
}
IN_AUTHOR_BIPARTITE = NYT_DIR / "ice_bipartite_author_keyword.gexf"
IN_AUTHOR_PROJECTION = NYT_DIR / "ice_author_projection.gexf"
PERIOD_LABELS = ["2003-2016", "2017-2020", "2021-2024", "2025-presente"]

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("community_detection")
pd.set_option("display.max_colwidth", 60)
print(f"BASE_DIR: {BASE_DIR}")

## 1. Caricamento delle reti

Per ogni giornale carichiamo la rete completa, quella diluita (archi con peso ≥ 5, notebook 02/08) e le quattro reti per periodo. Nei nodi c'è già `count`, il numero di articoli in cui la keyword compare, che serve per normalizzare i pesi in §2. Serve anche il numero di articoli del corpus da cui la rete è stata costruita: 2.093 articoli ICE del NYT, gli articoli Guardian con `is_ice_relevant`.

Nel Guardian l'id del nodo è il tag (`us-news/us-politics`), il nome leggibile è nell'attributo `label`: nelle tabelle usiamo sempre `label`.

Controllo di coerenza: nessuna keyword può comparire in più articoli di quelli del corpus (`count` ≤ articoli) e nessun arco può pesare più della keyword meno frequente dei due estremi.

**Pulizia delle keyword.** Le reti NYT sono quelle del notebook 02 dopo la correzione del 2026-10-05: keyword duplicate nello stesso articolo tolte (gonfiavano `count` e i pesi di alcuni archi e creavano 4 self-loop) e tag tecnici del sistema editoriale NYT esclusi (`internal-open-access-from-nl`, `live-detached`, `content type: personal profile`...). Dettagli in `docs/topic_network.md`. Il controllo sotto deve quindi dare 0 violazioni.


In [ ]:
def load_n_articles(paper: str) -> int:
    df = pd.read_parquet(PAPERS[paper]["articles"])
    return int(df["is_ice_relevant"].sum()) if "is_ice_relevant" in df else len(df)


def load_keyword_graphs(paper: str) -> dict[str, nx.Graph]:
    """{'full', 'pruned', <periodo>...} -> grafo di co-occorrenza letto dal GEXF."""
    base = PAPERS[paper]["dir"]
    graphs = {
        "full": nx.read_gexf(base / "ice_topic_network.gexf"),
        "pruned": nx.read_gexf(base / "ice_topic_network_pruned.gexf"),
    }
    for label in PERIOD_LABELS:
        graphs[label] = nx.read_gexf(base / f"ice_topic_network_{label}.gexf")
    return graphs


def check_counts(graph: nx.Graph, n_articles: int) -> int:
    """Numero di violazioni: count > articoli oppure peso dell'arco > count minimo degli estremi."""
    bad_nodes = sum(1 for _, c in graph.nodes(data="count") if c > n_articles)
    bad_edges = sum(1 for u, v, w in graph.edges(data="weight")
                    if w > min(graph.nodes[u]["count"], graph.nodes[v]["count"]))
    return bad_nodes + bad_edges


def summary_row(paper: str, name: str, graph: nx.Graph) -> dict:
    # Variabile con il numero di nodi della componente con più nodi connessi
    giant = max((len(c) for c in nx.connected_components(graph)), default=0)
    return {
        "giornale": paper, "rete": name,
        "nodi": graph.number_of_nodes(), "archi": graph.number_of_edges(),
        "densità": round(nx.density(graph), 3),
        "componenti": nx.number_connected_components(graph),
        "nodi_componente_maggiore": giant,
    }


N_ARTICLES = {paper: load_n_articles(paper) for paper in PAPERS}
KEYWORD_GRAPHS = {paper: load_keyword_graphs(paper) for paper in PAPERS}

# I grafi per periodo hanno `count` calcolato sugli articoli del periodo: il controllo vale solo per full/pruned
for paper, graphs in KEYWORD_GRAPHS.items():
    for name in ["full", "pruned"]:
        violations = check_counts(graphs[name], N_ARTICLES[paper])
        log.info("%s/%s: %d violazioni di coerenza", paper, name, violations)

B_authors = nx.read_gexf(IN_AUTHOR_BIPARTITE)
n_author_nodes = sum(1 for _, b in B_authors.nodes(data="bipartite") if b == 0)
log.info("Articoli: %s", N_ARTICLES)
log.info("Bipartito autore-keyword: %d autori, %d keyword, %d archi",
         n_author_nodes, B_authors.number_of_nodes() - n_author_nodes, B_authors.number_of_edges())

pd.DataFrame([
    summary_row(paper, name, g) for paper, graphs in KEYWORD_GRAPHS.items() for name, g in graphs.items()
])

**Come leggere la tabella.** Una riga per rete. `densità` = quota delle coppie di nodi collegate da un arco. `componenti` = pezzi della rete non collegati tra loro; `nodi_componente_maggiore` = nodi nel pezzo più grande. Le comunità si cercano sulla componente maggiore: i pezzi staccati sono comunità ovvie (pochi nodi isolati), non informative. Le reti per periodo non sono diluite: in §7 servirà decidere come trattarle (il 2021-2024 ha solo 96 articoli NYT e 90 Guardian, una soglia di peso 5 lo svuoterebbe).

## 2. Pesi delle reti di keyword: association strength

**Il problema.** Il peso di un arco è il numero di articoli in cui due keyword compaiono insieme. Le keyword molto frequenti ("illegal immigration", "Trump, Donald J") compaiono con tutto, quindi hanno co-occorrenze alte con quasi tutte le altre anche quando il legame non è specifico. Con i conteggi grezzi gli archi più forti sono sempre quelli tra hub (tabella sotto).

**La normalizzazione.** Usiamo l'**association strength** (van Eck & Waltman 2009, la normalizzazione di VOSviewer):

$$AS_{ij} = \frac{c_{ij} \cdot N}{s_i \cdot s_j}$$

con $c_{ij}$ = articoli con entrambe le keyword, $s_i$, $s_j$ = articoli con ciascuna keyword, $N$ = articoli del corpus. $s_i s_j / N$ è il numero di co-occorrenze atteso se le due keyword fossero indipendenti, quindi $AS$ = osservate / attese: **1** = come il caso, **> 1** = insieme più del caso, **< 1** = meno del caso.

Perché questa e non altre:

- **Jaccard** ($c_{ij} / (s_i + s_j - c_{ij})$) penalizza ogni coppia con una keyword molto più frequente dell'altra, anche quando la keyword rara compare *sempre* insieme a quella frequente: è un legame forte che Jaccard schiaccia vicino a 0.
- **PMI** è il logaritmo di $AS$: stesso ordinamento, ma dà pesi negativi alle coppie sotto il caso, che gli algoritmi di community detection non accettano.
- $AS$ è la misura per cui è stato studiato il legame con la funzione di qualità usata in §4 (Waltman, van Eck & Noyons 2010), e quella usata in bibliometria per le mappe di co-occorrenza.

Il peso originale resta nell'attributo `weight`; il nuovo è in `assoc`.

In [ ]:
# Crea un nuovo peso per gli archi chiamato `assoc`, cioè il rapporto tra le co-occorrenze osservate e quelle attese dal caso 
def add_association_strength(graph: nx.Graph, n_articles: int) -> nx.Graph:
    """Copia del grafo con attributo `assoc` = co-occorrenze osservate / attese sugli archi."""
    out = graph.copy()
    for u, v, data in out.edges(data=True):
        data["assoc"] = data["weight"] * n_articles / (out.nodes[u]["count"] * out.nodes[v]["count"])
    return out


def node_label(graph: nx.Graph, node: str) -> str:
    return graph.nodes[node].get("label", node)


def edge_table(graph: nx.Graph) -> pd.DataFrame:
    return pd.DataFrame(
        [(node_label(graph, u), node_label(graph, v), d["weight"], d["assoc"]) for u, v, d in graph.edges(data=True)],
        columns=["keyword_a", "keyword_b", "conteggio", "assoc"],
    )


ASSOC_GRAPHS = {
    paper: {name: add_association_strength(graphs[name], N_ARTICLES[paper]) for name in ["full", "pruned"]}
    for paper, graphs in KEYWORD_GRAPHS.items()
}

# Prendiamo diversi quantili e vediamo qual'è l'associazione più forte in assoluto nella rete
# Sotto_il_caso_pct è la percentuale di archi con associazione < 1, cioè quando le due parole compaiono insieme meno 
# spesso di quanto ci si aspetterebbe se fossero combinate a caso. In sostanza dice quale percentuale della rete rappresenta
# rumore o associazioni deboli che andrebbero tagliate.

quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]
rows = []
for paper, graphs in ASSOC_GRAPHS.items():
    for name, g in graphs.items():
        a = edge_table(g)["assoc"]
        rows.append({"giornale": paper, "rete": name,
                     **{f"q{int(q * 100)}": round(a.quantile(q), 1) for q in quantiles},
                     "max": round(a.max(), 1), "sotto_il_caso_pct": round((a < 1).mean() * 100, 1)})
pd.DataFrame(rows)

**Come leggere la tabella.** Distribuzione di `assoc` sugli archi di ciascuna rete (quantili e massimo); `sotto_il_caso_pct` = archi con keyword che compaiono insieme meno di quanto farebbero per caso.

Nella rete **completa** la mediana è alta e il massimo è uguale al numero di articoli: sono le coppie di keyword che compaiono in un solo articolo, e solo lì insieme ($c = s_i = s_j = 1$, quindi $AS = N$). Una coincidenza in un articolo diventa il legame più forte della rete. Nella rete **diluita** non succede: con $c_{ij} \geq 5$ il massimo possibile è $N / 5$, e gli archi casuali sono già stati tolti dalla soglia sul peso. Per questo l'association strength si usa solo dopo aver tolto le keyword rare (VOSviewer chiede un numero minimo di occorrenze per keyword).

In [ ]:
N_TOP_EDGES = 15
N_HUBS = 5

#costruiamo la edge_table con il count e l'assoc di ogni arco
#troviamo i 5 hub (keyword con grado pesato più alto)
#prendiamo i primi 15 archi per conteggio e per assoc
#contiamo quanti di questi archi toccano uno degli hub (grado pesato)

def top_edges_comparison(graph: nx.Graph) -> tuple[pd.DataFrame, dict]:
    """Archi più forti per conteggio e per assoc, e quanti dei primi toccano uno degli hub (grado pesato)."""
    edges = edge_table(graph)
    hubs = {node_label(graph, n) for n, _ in sorted(graph.degree(weight="weight"), key=lambda x: -x[1])[:N_HUBS]}
    touches_hub = lambda df: int((df["keyword_a"].isin(hubs) | df["keyword_b"].isin(hubs)).sum())
    by_count = edges.nlargest(N_TOP_EDGES, "conteggio").reset_index(drop=True)
    by_assoc = edges.nlargest(N_TOP_EDGES, "assoc").reset_index(drop=True)
    side = pd.concat({
        "per conteggio": by_count.assign(coppia=by_count["keyword_a"] + "  —  " + by_count["keyword_b"])[["coppia", "conteggio", "assoc"]],
        "per association strength": by_assoc.assign(coppia=by_assoc["keyword_a"] + "  —  " + by_assoc["keyword_b"])[["coppia", "conteggio", "assoc"]],
    }, axis=1).round(1)
    return side, {"hub": sorted(hubs), "top_per_conteggio_con_hub": touches_hub(by_count), "top_per_assoc_con_hub": touches_hub(by_assoc)}


for paper in PAPERS:
    side, hub_info = top_edges_comparison(ASSOC_GRAPHS[paper]["pruned"])
    print(f"=== {paper}, rete diluita — hub: {', '.join(hub_info['hub'])}")
    print(f"archi tra i primi {N_TOP_EDGES} che toccano un hub: per conteggio {hub_info['top_per_conteggio_con_hub']}, "
          f"per association strength {hub_info['top_per_assoc_con_hub']}")
    display(side)

**Come leggere le tabelle.** A sinistra i 15 archi più forti per conteggio, a destra per association strength, sulla rete diluita; gli hub sono le 5 keyword con il grado pesato più alto. Per conteggio i primi archi sono tutti tra hub e dicono poco (che "Trump" e "illegal immigration" stiano insieme in un corpus su ICE è scontato). Per association strength emergono legami specifici, spesso una singola vicenda raccontata in più articoli (un caso locale, una persona e il luogo in cui è avvenuto il fatto): sono questi legami che tengono insieme le comunità tematiche.

Nel Guardian una parte dei legami più forti è tra tag quasi sinonimi o annidati (`Television` / `Television & radio` / `TV comedy` / `Comedy`, `Denver` / `Colorado`): il Guardian assegna spesso sia il tag generale sia quello specifico. Formeranno comunità molto compatte ma ovvie, da riconoscere in lettura.

Attenzione all'altro estremo: $AS$ premia le coppie di keyword rare che compaiono sempre insieme. Nella rete diluita servono almeno 5 articoli in comune, quindi sono vicende vere, ma una comunità potrebbe ridursi a "una notizia". È uno dei motivi per cui in §4 si sceglie la risoluzione guardando la dimensione delle comunità, non solo la qualità della partizione.

### 2.1 Rete di controllo: soglia sulle keyword invece che sugli archi

La rete diluita dipende dalla soglia `MIN_EDGE_WEIGHT = 5` scelta a mano nel notebook 02. Come controllo usiamo il filtro di VOSviewer: si tolgono le keyword rare (meno di `MIN_OCCURRENCES` articoli) e si tengono **tutti** gli archi tra quelle rimaste, anche di peso 1. Se le comunità delle due reti si somigliano (§5), il risultato non dipende da come è stato tolto il rumore.

Sotto: dimensioni della rete di controllo a diverse soglie, per scegliere `MIN_OCCURRENCES`.

In [ ]:
def occurrence_filtered(graph: nx.Graph, min_occurrences: int) -> nx.Graph:
    """Sottografo delle keyword presenti in almeno min_occurrences articoli (tutti gli archi tra loro), senza nodi isolati."""
    kept = [n for n, c in graph.nodes(data="count") if c >= min_occurrences]
    sub = graph.subgraph(kept).copy()
    sub.remove_nodes_from(list(nx.isolates(sub)))
    return sub


rows = []
for paper in PAPERS:
    full = ASSOC_GRAPHS[paper]["full"]
    for t in [3, 5, 8, 10]:
        sub = occurrence_filtered(full, t)
        a = np.array([w for _, _, w in sub.edges(data="assoc")])
        rows.append({**summary_row(paper, f"keyword ≥ {t} articoli", sub),
                     "archi_peso_1_pct": round(np.mean([w == 1 for _, _, w in sub.edges(data="weight")]) * 100, 1),
                     "assoc_max": round(a.max(), 1)})
    rows.append({**summary_row(paper, "diluita (peso ≥ 5)", ASSOC_GRAPHS[paper]["pruned"]),
                 "archi_peso_1_pct": 0.0,
                 "assoc_max": round(max(w for _, _, w in ASSOC_GRAPHS[paper]["pruned"].edges(data="assoc")), 1)})
pd.DataFrame(rows)

**Come leggere la tabella.** Per ogni giornale, la rete di controllo a quattro soglie sulle keyword e, nell'ultima riga, la rete diluita con cui confrontarla. `archi_peso_1_pct` = quota di archi che vengono da un solo articolo; `assoc_max` = association strength massima (con soglia $t$ non può superare $N / t$).

**Scelta: `MIN_OCCURRENCES = 5`.** È la soglia a cui la rete di controllo ha circa gli stessi nodi della diluita (NYT 434 contro 369, Guardian 177 contro 172), quindi il confronto in §5 è tra due reti sugli stessi temi che differiscono solo per gli archi. È anche il valore predefinito di VOSviewer. Le due reti restano molto diverse negli archi: quella di controllo ne ha 5 volte tanti (NYT 12.890 contro 2.409) e circa metà vengono da un solo articolo; l'association strength massima è la stessa ($N / 5$).

In [ ]:
MIN_OCCURRENCES = 5
for paper in PAPERS:
    ASSOC_GRAPHS[paper]["control"] = occurrence_filtered(ASSOC_GRAPHS[paper]["full"], MIN_OCCURRENCES)
    g = ASSOC_GRAPHS[paper]["control"]
    log.info("%s, rete di controllo (keyword >= %d articoli): %d nodi, %d archi",
             paper, MIN_OCCURRENCES, g.number_of_nodes(), g.number_of_edges())

## 3. Rete degli autori: similarità e backbone

**Il problema della proiezione attuale.** Nel notebook 03 due autori sono collegati con peso = numero di keyword condivise (almeno 3). Le keyword generiche ("illegal immigration", "deportation") sono usate da quasi tutti, quindi collegano quasi tutti: sui 358 autori inclusi la proiezione ha densità 0,81. E chi firma molti articoli usa molte keyword, quindi condivide molte keyword con tutti: il peso misura soprattutto la produttività. Una community detection su questa rete troverebbe "chi scrive tanto", non "chi scrive degli stessi temi".

**Cosa facciamo** (tre passi):

1. **Autori inclusi**: solo chi ha firmato almeno `MIN_AUTHOR_ARTICLES = 2` articoli ICE (§3.1).
2. **Profilo e similarità** (§3.2): ogni autore è un vettore di keyword pesato con **TF-IDF**; due autori sono collegati con peso = **similarità coseno** dei profili.
3. **Backbone** (§3.3): il **disparity filter** tiene, per ogni autore, solo gli archi significativamente più forti degli altri suoi archi.

**Copertura**: tutto il 2003-2026. Dal 2025 il NYT non compila più `byline.person`, ma la firma resta nel testo `byline.original` (98% degli articoli 2025+): gli autori si leggono da lì, come nel notebook 03. Fino al 2026-10-05 questa lettura mancava e la rete copriva solo il 2003-2024 (117 autori invece di 358).

### 3.1 Autori, profili e correzioni dei dati

I profili degli autori vengono ricostruiti qui dagli articoli (`articles_ice.parquet`). Controllandoli sono emersi tre problemi dei dati NYT, corretti qui e nel bipartito del notebook 03 (vedi `docs/bipartite_network.md`):

1. **Firme senza cognome.** In alcuni articoli `byline.person` ha `lastname` vuoto: Michael D. Shear compare come "Michael" in 16 articoli (e nel notebook 03 è un nodo "Michael"), e lo stesso accade una volta per Scott M. Stringer, Robert M. Morgenthau, Andrew M. Cuomo e Julián Aguilar. Il nome completo si recupera dalla firma testuale (`byline.original`, "By Michael D. Shear and Ron Nixon"): si cerca il nome che inizia con il `firstname` registrato; se non c'è un'unica corrispondenza il nome resta com'è.
2. **Il nome dell'autore tra le keyword.** In una parte degli articoli firmati il NYT mette tra le keyword `persons` il nome di chi firma ("preston, julia", "nixon, ron"); capita anche negli editoriali scritti da politici ("escobar, veronica"). Queste keyword identificano l'autore, non un tema: sono escluse dai profili tutte le keyword che iniziano con "cognome, nome" di un autore del corpus.

3. **Firme del 2025+ solo testuali** (corretto il 2026-10-05). Quando `byline.person` è vuoto, gli autori si leggono da `byline.original`: si separa su virgole e "and", si toglie "By" ("Por" nelle firme in spagnolo) e si scartano firme istituzionali ("The Editorial Board", "The New York Times") e fotografi. Due giornalisti scritti in modo diverso nei due schemi sono uniti con `AUTHOR_ALIASES` (Jonah E. Bromwich, Orlando Mayorquín).

Le keyword sono estratte con la stessa logica di `extract_keywords` del notebook 02 (duplicati e tag tecnici esclusi, §1).

In [ ]:
MIN_AUTHOR_ARTICLES = 2
ICE_KEYWORD = "immigration and customs enforcement (us)"
# tag del sistema editoriale NYT finiti tra le keyword `subject` dal 2025 (es. "internal-open-access-from-nl", "audio-neutral-informative")
TECHNICAL_TAG_RE = re.compile(r"^(internal-|content type:|live-detached$|audio-|vis-|your-feed-)")
KEYWORD_NAME_ALIASES = {
    "subject": "subject", "organizations": "organizations", "organization": "organizations",
    "persons": "persons", "person": "persons", "glocations": "glocations", "location": "glocations",
    "creative_works": "creative_works", "title": "creative_works",
}


def extract_keywords(keywords: list) -> list[str]:
    """Keyword di UN articolo come nel notebook 02 (stesse categorie, ICE e tag tecnici esclusi, senza duplicati)."""
    values = []
    for kw in keywords or []:
        if KEYWORD_NAME_ALIASES.get(kw.get("name", "").strip().lower()):
            value = kw.get("value", "").strip().lower()
            if value and value != ICE_KEYWORD and value not in values and not TECHNICAL_TAG_RE.match(value):
                values.append(value)
    return values


# come nel notebook 03: voci della firma testuale che non sono giornalisti, e stesso autore scritto in due modi
NON_AUTHOR_RE = re.compile(r"^(the |via |photographs by )", re.IGNORECASE)
AUTHOR_ALIASES = {"Jonah Engel Bromwich": "Jonah E. Bromwich", "Orlando Mayorquin": "Orlando Mayorquín"}


def normalize_name(parts: list[str]) -> str:
    name = " ".join(x.title() if x.isupper() else x for x in parts)
    return AUTHOR_ALIASES.get(name, name)


def extract_authors(byline: dict, repairs: Counter) -> list[str]:
    """Come notebook 03: da byline.person; se manca il cognome, nome completo da byline.original;
    dal 2025 (byline.person vuoto) tutti gli autori dalla firma testuale byline.original."""
    text = re.sub(r"^(By|Por)\s+", "", html.unescape(byline.get("original") or "").strip(), flags=re.IGNORECASE)
    names_in_text = [n.strip() for n in re.split(r",\s*|\s+and\s+", text) if n.strip()]
    if not byline.get("person"):
        return list(dict.fromkeys(normalize_name(n.split()) for n in names_in_text if not NON_AUTHOR_RE.match(n)))
    authors = []
    for p in byline.get("person") or []:
        parts = [x.strip() for x in [p.get("firstname"), p.get("middlename"), p.get("lastname")] if x and x.strip()]
        if not parts:
            continue
        if not (p.get("lastname") or "").strip():
            matches = [n for n in names_in_text if n.lower().startswith(parts[0].lower())]
            if len(matches) == 1:
                repairs[(normalize_name(parts), normalize_name(matches[0].split()))] += 1
                parts = matches[0].split()
        authors.append(normalize_name(parts))
    return list(dict.fromkeys(authors))


def load_author_articles() -> tuple[pd.DataFrame, Counter]:
    """Una riga per coppia (autore, articolo) con le keyword dell'articolo."""
    df = pd.read_parquet(PAPERS["nyt"]["articles"], columns=["id", "byline_json", "keywords_json"])
    repairs = Counter()
    df["author"] = [extract_authors(json.loads(b) or {}, repairs) for b in df["byline_json"]]
    df["keywords"] = [extract_keywords(json.loads(k)) for k in df["keywords_json"]]
    return df[["id", "author", "keywords"]].explode("author").dropna(subset=["author"]).reset_index(drop=True), repairs


def author_name_keys(authors) -> set[str]:
    """'cognome, nome' in minuscolo per ogni autore con almeno due parole nel nome."""
    return {f"{a.split()[-1].lower()}, {a.split()[0].lower()}" for a in authors if len(a.split()) > 1}


author_rows, name_repairs = load_author_articles()
author_articles = author_rows.groupby("author")["id"].nunique().sort_values(ascending=False).rename("articoli")
display(pd.DataFrame([{"nome registrato": a, "nome recuperato": b, "articoli": n} for (a, b), n in name_repairs.items()]))

name_keys = author_name_keys(author_articles.index)
all_keywords = {k for kws in author_rows["keywords"] for k in kws}
name_keywords = sorted(k for k in all_keywords if any(k.startswith(key) for key in name_keys))
author_rows["keywords"] = author_rows["keywords"].apply(lambda kws: [k for k in kws if k not in set(name_keywords)])
log.info("Keyword uguali al nome di un autore, escluse dai profili: %d (%s, ...)", len(name_keywords), ", ".join(name_keywords[:6]))

bipartite_authors = {n for n, b in B_authors.nodes(data="bipartite") if b == 0}
log.info("Autori: %d (bipartito notebook 03: %d); solo qui: %s; solo nel bipartito: %s",
         len(author_articles), len(bipartite_authors),
         sorted(set(author_articles.index) - bipartite_authors), sorted(bipartite_authors - set(author_articles.index)))

rows = []
for t in [1, 2, 3, 5]:
    kept = author_articles[author_articles >= t]
    rows.append({
        "articoli_minimi": t, "autori": len(kept),
        "autori_pct": round(len(kept) / len(author_articles) * 100, 1),
        "firme_coperte_pct": round(kept.sum() / author_articles.sum() * 100, 1),
        "keyword_nei_profili": len({k for kws in author_rows.loc[author_rows["author"].isin(kept.index), "keywords"] for k in kws}),
    })
display(pd.DataFrame(rows).set_index("articoli_minimi"))

authors = sorted(author_articles[author_articles >= MIN_AUTHOR_ARTICLES].index)
log.info("Autori inclusi (>= %d articoli): %d", MIN_AUTHOR_ARTICLES, len(authors))

**Come leggere la seconda tabella.** Una riga per soglia. `autori_pct` = quota degli 819 autori che resta; `firme_coperte_pct` = quota delle firme (coppie autore-articolo) che appartiene agli autori rimasti: dice quanto del lavoro giornalistico resta nella rete anche se molti autori escono; `keyword_nei_profili` = keyword usate da almeno un autore rimasto.

**Come leggere la prima tabella.** Le firme riparate: nome registrato dal NYT, nome completo recuperato, articoli coinvolti. Il log sotto elenca le keyword-nome escluse e confronta gli autori con quelli del bipartito del notebook 03 (dopo la correzione del notebook 03 non devono esserci differenze).

**Scelta: `MIN_AUTHOR_ARTICLES = 2`**: è la soglia minima per cui un profilo descrive un autore e non un articolo. Restano 358 autori su 819 (44%) ma l'85% delle firme: escono soprattutto autori di un solo articolo (molti coautori occasionali degli articoli a più firme del 2025-2026).

### 3.2 Profili TF-IDF e similarità coseno

**Profilo**: per ogni autore, un vettore con una posizione per keyword; il valore è il numero di articoli dell'autore con quella keyword, moltiplicato per l'**IDF** della keyword:

$$\text{idf}_k = \log \frac{A}{a_k}$$

con $A$ = autori inclusi e $a_k$ = autori che hanno usato la keyword $k$. Una keyword usata da tutti gli autori ha IDF 0 e non conta; una usata da pochi conta molto. È la stessa idea dell'association strength in §2: togliere il peso di ciò che è comune a tutti.

**Similarità coseno** tra due profili: 1 = stesse keyword nelle stesse proporzioni, 0 = nessuna keyword in comune (dopo l'IDF). Dipende dalla *composizione* del profilo, non dalla sua lunghezza: un autore con 70 articoli e uno con 2 possono avere similarità alta se scrivono degli stessi temi. La rete di similarità ha un arco per ogni coppia con coseno > 0.

Controllo sotto: confronto con la proiezione del notebook 03 sugli stessi autori. La correlazione di Spearman tra la **forza** di un autore (somma dei pesi dei suoi archi) e il numero di articoli che ha firmato misura quanto la rete dipende dalla produttività: dovrebbe essere alta nella proiezione e scendere nella rete di similarità.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def author_profiles(author_rows: pd.DataFrame, authors: list[str]) -> pd.DataFrame:
    """Matrice autori x keyword, valore = articoli dell'autore con quella keyword."""
    pairs = author_rows[author_rows["author"].isin(authors)].explode("keywords").dropna(subset=["keywords"]).reset_index(drop=True)
    return pd.crosstab(pairs["author"], pairs["keywords"])


def tfidf(profiles: pd.DataFrame) -> pd.DataFrame:
    idf = np.log(len(profiles) / (profiles > 0).sum())
    return profiles * idf


def similarity_graph(weighted: pd.DataFrame) -> nx.Graph:
    """Un arco per ogni coppia di autori con similarità coseno > 0 (peso = coseno)."""
    sim = cosine_similarity(weighted.values)
    names = weighted.index.tolist()
    graph = nx.Graph()
    graph.add_nodes_from(names)
    i_idx, j_idx = np.triu_indices(len(names), k=1)
    graph.add_weighted_edges_from(
        (names[i], names[j], float(sim[i, j])) for i, j in zip(i_idx, j_idx) if sim[i, j] > 0
    )
    return graph


def productivity_correlation(graph: nx.Graph) -> float:
    strength = pd.Series(dict(graph.degree(weight="weight")))
    return round(strength.corr(author_articles.reindex(strength.index), method="spearman"), 2)


profiles = author_profiles(author_rows, authors)
profiles_tfidf = tfidf(profiles)
G_similarity = similarity_graph(profiles_tfidf)

old_projection = nx.read_gexf(IN_AUTHOR_PROJECTION).subgraph(authors).copy()
old_projection.add_nodes_from(a for a in authors if a not in old_projection)

pd.DataFrame([
    {**summary_row("nyt", "", old_projection), "rete": "proiezione notebook 03 (keyword condivise >= 3)",
     "spearman_forza_articoli": productivity_correlation(old_projection)},
    {**summary_row("nyt", "", G_similarity), "rete": "similarità coseno TF-IDF",
     "spearman_forza_articoli": productivity_correlation(G_similarity)},
]).drop(columns=["giornale"]).set_index("rete")

**Come leggere la tabella.** Le due reti sugli stessi autori. `spearman_forza_articoli` vicino a 1 = gli autori più collegati sono semplicemente quelli che scrivono di più; vicino a 0 = la posizione nella rete non dipende dalla produttività. La rete di similarità è ancora densa (basta una keyword in comune non universale per avere un arco): è il backbone in §3.3 a renderla leggibile.

**Risultato.** La correlazione con la produttività scende (0,72 → 0,54), ma non sparisce: il coseno toglie l'effetto della lunghezza del profilo *sul singolo arco*, ma un autore con molti articoli copre più temi e ha similarità moderate con più colleghi, quindi la sua forza (somma di tutti gli archi) resta più alta. Nella rete di similarità quasi completa (densità 0,97) la forza è in pratica la similarità media con tutti gli altri. Il backbone in §3.3 riduce questa dipendenza.

In [ ]:
N_SHOWN_AUTHORS = 8

rows = []
for author in author_articles.loc[authors].sort_values(ascending=False).index[:N_SHOWN_AUTHORS]:
    neighbors = sorted(G_similarity[author].items(), key=lambda x: -x[1]["weight"])[:3]
    rows.append({
        "autore": author, "articoli": author_articles[author],
        "keyword_caratterizzanti": ", ".join(profiles_tfidf.loc[author].nlargest(5).index),
        "autori_più_simili": ", ".join(f"{n} ({d['weight']:.2f})" for n, d in neighbors),
    })
pd.DataFrame(rows).set_index("autore")

**Come leggere la tabella.** Controllo a campione sugli autori con più articoli: le 5 keyword con il TF-IDF più alto nel loro profilo (i temi che li distinguono dagli altri autori, non i più frequenti) e i 3 autori più simili con il coseno tra parentesi. Se il metodo funziona, gli autori simili condividono temi riconoscibili (lo stesso filone di inchiesta, la stessa area geografica).

**Risultato.** Il controllo è superato: i profili colgono filoni riconoscibili. Julia Preston (2003-2016: Obama, criminalità) è vicina a Kirk Semple e Randal C. Archibold; Miriam Jordan e Caitlin Dickerson (famiglie e bambini separati) sono reciprocamente le più simili; Luis Ferré-Sadurní e Ana Ley (New York, Rikers, Delaney Hall) sono vicini ai cronisti di New York e New Jersey; Ernesto Londoño a Mitch Smith (Minneapolis, coseno 0,77). Hamed Aleaziz, l'autore con più articoli ICE (173, quasi tutti 2025-2026), ha un profilo generico (DHS, deportazioni, Homan, Trump): è simile a molti colleghi ma a nessuno in modo forte.

### 3.3 Backbone: disparity filter

La rete di similarità collega quasi tutti con archi deboli. Una soglia globale sul coseno (es. tenere gli archi > 0,2) taglierebbe soprattutto gli autori con profili ampi, che hanno similarità più basse con tutti. Il **disparity filter** (Serrano, Boguñá & Vespignani 2009) giudica ogni arco rispetto agli altri archi dello stesso nodo.

Per un autore $i$ con $k_i$ archi e forza $s_i$, la quota di forza su un arco è $p_{ij} = w_{ij} / s_i$. Se la forza fosse distribuita a caso tra i $k_i$ archi, la probabilità di vedere una quota almeno così grande sarebbe

$$\alpha_{ij} = (1 - p_{ij})^{k_i - 1}$$

L'arco è tenuto se $\alpha_{ij} < \alpha$ per **almeno uno** dei due autori: è un legame che spicca per almeno uno dei due. Gli autori con un solo arco non possono giudicare quell'arco (resta se è significativo per l'altro autore).

Sotto: il backbone a tre livelli di $\alpha$. Gli autori rimasti senza archi sono autori senza nessun legame che spicchi: in §6 non avranno comunità.

In [ ]:
def disparity_filter(graph: nx.Graph, alpha: float) -> nx.Graph:
    """Backbone: archi con alpha_ij < alpha per almeno un estremo. Conserva tutti i nodi (anche isolati)."""
    kept = set()
    for i in graph:
        k = graph.degree(i)
        if k < 2:
            continue
        s = graph.degree(i, weight="weight")
        for j, data in graph[i].items():
            if (1 - data["weight"] / s) ** (k - 1) < alpha:
                kept.add(frozenset((i, j)))
    backbone = nx.Graph()
    backbone.add_nodes_from(graph.nodes(data=True))
    backbone.add_edges_from((u, v, graph[u][v]) for u, v in map(tuple, kept))
    return backbone


total_weight = G_similarity.size(weight="weight")
rows = []
for alpha in [0.01, 0.05, 0.1]:
    bb = disparity_filter(G_similarity, alpha)
    connected = bb.subgraph([n for n in bb if bb.degree(n) > 0])
    rows.append({
        "alpha": alpha, "archi": bb.number_of_edges(),
        "archi_pct": round(bb.number_of_edges() / G_similarity.number_of_edges() * 100, 1),
        "peso_pct": round(bb.size(weight="weight") / total_weight * 100, 1),
        "autori_isolati": sum(1 for n in bb if bb.degree(n) == 0),
        "componenti": nx.number_connected_components(connected),
        "nodi_componente_maggiore": max(len(c) for c in nx.connected_components(connected)),
        "densità": round(nx.density(bb), 3),
        "spearman_forza_articoli": productivity_correlation(bb),
    })
pd.DataFrame(rows).set_index("alpha")

**Come leggere la tabella.** Una riga per livello di significatività. `archi_pct` e `peso_pct` = quota di archi e di similarità totale conservata; `autori_isolati` = autori rimasti senza archi; `componenti` e `nodi_componente_maggiore` sono calcolati senza gli isolati. α più basso = criterio più severo = rete più sparsa.

**Scelta: `BACKBONE_ALPHA = 0.05`**, il valore usato da Serrano et al. e il più comune in letteratura. Tiene il 10% degli archi ma il 43% della similarità totale (restano gli archi forti), nessun autore isolato e una sola componente; la densità scende da 0,97 a 0,09 e la correlazione con la produttività da 0,54 a 0,34. Con α = 0,01 la rete è più sparsa (densità 0,04) e ancora meno legata alla produttività (Spearman 0,21), sempre senza autori isolati: in §6 la useremo come controllo, per vedere se le comunità cambiano.

La dipendenza dalla produttività non sparisce del tutto: gli autori con più articoli hanno più legami significativi. È in parte reale (chi copre ICE per anni incrocia più colleghi), da tenere presente leggendo le comunità.

In [ ]:
BACKBONE_ALPHA = 0.05
G_authors_backbone = disparity_filter(G_similarity, BACKBONE_ALPHA)
G_authors_control = disparity_filter(G_similarity, 0.01)
log.info("Backbone autori (alpha=%.2f): %d nodi, %d archi; controllo (alpha=0.01): %d archi",
         BACKBONE_ALPHA, G_authors_backbone.number_of_nodes(), G_authors_backbone.number_of_edges(),
         G_authors_control.number_of_edges())

## 4. Algoritmo (Leiden), risoluzione e stabilità

**Leiden** (Traag, Waltman & van Eck 2019) cerca la partizione dei nodi che massimizza una *funzione di qualità*. È il successore di Louvain: stessa idea (spostare nodi tra comunità finché la qualità migliora), più un passo di rifinitura che garantisce comunità connesse al loro interno, cosa che Louvain non garantisce. Implementazione: `leidenalg` su `igraph`.

La funzione di qualità decide che cosa conta come "comunità". Due candidate:

- **Modularità** (Newman 2006): confronta il peso degli archi dentro una comunità con quello atteso se gli archi fossero distribuiti a caso rispettando la forza di ogni nodo. Una comunità è un gruppo di nodi più legati tra loro di quanto ci si aspetterebbe dato quanto sono collegati in generale.
- **CPM** (Constant Potts Model, Traag et al. 2011): una comunità è un gruppo in cui il peso medio tra *tutte* le coppie di nodi (anche quelle senza arco, che valgono 0) supera una soglia fissa γ. Non ha un modello nullo: è la funzione usata da VOSviewer insieme all'association strength.

Entrambe hanno un **parametro di risoluzione** γ: più alto = comunità più piccole e numerose. Non esiste un valore "giusto"; si sceglie in §4.2 guardando stabilità e dimensioni delle comunità.

**Non-determinismo.** Leiden parte da un ordine casuale dei nodi: seed diversi possono dare partizioni diverse. Ogni configurazione è eseguita con `N_SEEDS` seed e la somiglianza tra le partizioni è misurata con l'**AMI** (adjusted mutual information, come per BERTopic nei notebook 04/07/09): 1 = partizioni identiche, 0 = somiglianti quanto due partizioni casuali.

Le comunità si cercano sulla **componente maggiore** di ogni rete: i pezzi staccati (2-3 nodi) sono comunità ovvie e non informative.

In [ ]:
import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_mutual_info_score

N_SEEDS = 10
MIN_COMMUNITY_SIZE = 5
QUALITY_FUNCTIONS = {"modularità": la.RBConfigurationVertexPartition, "CPM": la.CPMVertexPartition}


def largest_component(graph: nx.Graph) -> nx.Graph:
    """Componente maggiore con nodi e archi in ordine alfabetico.

    subgraph() di networkx itera su un set: l'ordine dei nodi cambierebbe a ogni processo (hash casuale
    di Python) e con lui il risultato di Leiden e delle reti casuali, anche a seed fisso.
    """
    nodes = sorted(max(nx.connected_components(graph), key=len))
    out = nx.Graph()
    out.add_nodes_from((n, graph.nodes[n]) for n in nodes)
    edges = [(min(u, v), max(u, v), d) for u, v, d in graph.subgraph(nodes).edges(data=True)]
    out.add_edges_from(sorted(edges, key=lambda e: e[:2]))
    return out


def to_igraph(graph: nx.Graph, weight: str) -> tuple[ig.Graph, list[str]]:
    """Grafo igraph con pesi in 'w' e lista dei nomi nell'ordine dei vertici."""
    nodes = list(graph)
    index = {n: i for i, n in enumerate(nodes)}
    g = ig.Graph(n=len(nodes), edges=[(index[u], index[v]) for u, v in graph.edges()])
    g.es["w"] = [d[weight] for _, _, d in graph.edges(data=True)]
    return g, nodes


def leiden(g: ig.Graph, quality: str, resolution: float, seed: int):
    return la.find_partition(g, QUALITY_FUNCTIONS[quality], weights="w",
                             resolution_parameter=resolution, seed=seed, n_iterations=-1)

# Serve per misurare stabilità algoritmo quando lo faccio girare più volte: abbiamo una lista con le soluzione trovate (membership) 
# confrontiamo ogni combinazione possibile e calcoliamo l'AMI che confronta le due partizioni. 
# ritorniamo poi la media di tutti i valori AMI calcolati. 
def mean_pairwise_ami(memberships: list[np.ndarray]) -> float:
    pairs = [(i, j) for i in range(len(memberships)) for j in range(i + 1, len(memberships))]
    return float(np.mean([adjusted_mutual_info_score(memberships[i], memberships[j]) for i, j in pairs]))


def describe_runs(partitions: list) -> dict:
    """Dimensioni delle comunità (mediana sui seed) e stabilità tra seed."""
    memberships = [np.array(p.membership) for p in partitions]
    sizes = [pd.Series(m).value_counts() for m in memberships]
    return {
        "comunità": int(np.median([len(s) for s in sizes])),
        f"comunità_≥{MIN_COMMUNITY_SIZE}": int(np.median([(s >= MIN_COMMUNITY_SIZE).sum() for s in sizes])),
        f"nodi_in_comunità_≥{MIN_COMMUNITY_SIZE}_pct": round(np.median([s[s >= MIN_COMMUNITY_SIZE].sum() / s.sum() for s in sizes]) * 100, 1),
        "comunità_maggiore_pct": round(np.median([s.max() / s.sum() for s in sizes]) * 100, 1),
        "modularità": round(float(np.median([p.modularity for p in partitions])), 3),
        "AMI_tra_seed": round(mean_pairwise_ami(memberships), 2),
    }

### 4.1 Pesi e funzione di qualità

Tre combinazioni sulle reti di keyword diluite, `N_SEEDS` seed ciascuna, risoluzione standard (γ = 1 per la modularità; per CPM γ = peso medio tra tutte le coppie di nodi, cioè la densità pesata della rete, il punto in cui CPM si comporta come la modularità su una rete casuale):

1. **conteggi + modularità**: la scelta "di default";
2. **association strength + modularità**;
3. **association strength + CPM**: la combinazione di VOSviewer.

Colonne: `comunità` (mediana sui seed), `comunità_≥5` e `nodi_in_comunità_≥5_pct` (le comunità di 1-4 nodi non si possono leggere come temi), `comunità_maggiore_pct` (quota di nodi nella comunità più grande: se è metà della rete, la partizione dice poco), `modularità` (della partizione trovata, calcolata con i pesi usati), `AMI_tra_seed`.

In [ ]:
rows = []
for paper in PAPERS:
    graph = largest_component(ASSOC_GRAPHS[paper]["pruned"])
    n = graph.number_of_nodes()
    weighted_density = 2 * graph.size(weight="assoc") / (n * (n - 1))
    for label, weight, quality, resolution in [
        ("conteggi + modularità", "weight", "modularità", 1.0),
        ("association strength + modularità", "assoc", "modularità", 1.0),
        ("association strength + CPM", "assoc", "CPM", weighted_density),
    ]:
        g, _ = to_igraph(graph, weight)
        runs = [leiden(g, quality, resolution, seed) for seed in range(N_SEEDS)]
        rows.append({"giornale": paper, "combinazione": label, "γ": round(resolution, 3), **describe_runs(runs)})
pd.DataFrame(rows).set_index(["giornale", "combinazione"])

**Come leggere la tabella.** Una riga per giornale e combinazione. Una buona partizione per i nostri scopi ha molte comunità di almeno 5 nodi, quasi tutti i nodi dentro comunità leggibili, nessuna comunità che si mangi la rete e AMI tra seed alto. La modularità **non** è confrontabile tra righe con pesi diversi (è calcolata su reti pesate in modo diverso).

**Risultato e scelta: association strength + modularità.**

- *Conteggi + modularità* dà poche comunità grandi (6-8, la maggiore con il 41% dei nodi del NYT e il 58% del Guardian) ed è la meno stabile (AMI 0,75-0,78): attorno agli hub si formano due o tre blocchi generici.
- *Association strength + CPM* è stabile (AMI 0,90-0,92) ma lascia **fuori** una parte grande della rete: le keyword-hub hanno association strength vicina a 1 con quasi tutte le altre (compaiono ovunque, quindi mai "più del caso"), nessun gruppo con loro supera la soglia γ e restano comunità di un solo nodo. Solo il 55-61% dei nodi finisce in comunità di almeno 5 keyword, e mancano proprio "deportation", "Trump", "illegal immigration".
- *Association strength + modularità* è il compromesso: comunità definite dai legami specifici, stabilità vicina a CPM (AMI 0,80-0,91), e ogni nodo, hub compresi, assegnato alla comunità a cui è più legato rispetto all'atteso. A γ = 1 resta però una comunità con quasi metà dei nodi: la si guarda in §4.2.

Nota: la modularità ha già un modello nullo che tiene conto della forza dei nodi, quindi con l'association strength la correzione per la frequenza è applicata due volte (sui pesi e nel modello nullo). È una scelta empirica, motivata dalla tabella; va dichiarata nel report.

Per la rete degli autori i pesi sono già normalizzati (coseno su profili TF-IDF, §3): si usa la modularità come per le keyword.

### 4.2 Risoluzione

Per ogni rete (le due reti di keyword per giornale, principale e di controllo, e le due reti degli autori) Leiden con modularità a diversi valori di γ, `N_SEEDS` seed ciascuno.

In [ ]:
RESOLUTIONS = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]

NETWORKS = {
    ("nyt", "keyword, diluita"): (ASSOC_GRAPHS["nyt"]["pruned"], "assoc"),
    ("nyt", "keyword, controllo"): (ASSOC_GRAPHS["nyt"]["control"], "assoc"),
    ("guardian", "keyword, diluita"): (ASSOC_GRAPHS["guardian"]["pruned"], "assoc"),
    ("guardian", "keyword, controllo"): (ASSOC_GRAPHS["guardian"]["control"], "assoc"),
    ("nyt", "autori, backbone α=0,05"): (G_authors_backbone, "weight"),
    ("nyt", "autori, controllo α=0,01"): (G_authors_control, "weight"),
}
NETWORKS = {key: (largest_component(graph), weight) for key, (graph, weight) in NETWORKS.items()}

resolution_runs = {}
rows = []
for key, (graph, weight) in NETWORKS.items():
    g, _ = to_igraph(graph, weight)
    for resolution in RESOLUTIONS:
        runs = [leiden(g, "modularità", resolution, seed) for seed in range(N_SEEDS)]
        resolution_runs[(*key, resolution)] = runs
        rows.append({"giornale": key[0], "rete": key[1], "nodi": graph.number_of_nodes(), "γ": resolution, **describe_runs(runs)})
resolution_table = pd.DataFrame(rows).set_index(["giornale", "rete", "γ"])
resolution_table

**Come leggere la tabella.** Un blocco per rete, una riga per γ. Al crescere di γ le comunità aumentano e si rimpiccioliscono; la modularità della partizione (sempre calcolata con γ = 1) cala quando le comunità diventano troppo piccole. Il criterio di scelta è l'`AMI_tra_seed` (una partizione che cambia da un seed all'altro non si può interpretare), insieme alla leggibilità: comunità abbastanza numerose da separare i temi, abbastanza grandi (≥ 5 nodi) da essere lette.

**La comunità "nucleo" delle reti diluite.** Nelle reti di keyword diluite, a γ basso, una comunità contiene il 45-70% dei nodi; nelle reti di controllo no. Controllata a mano (γ = 2): contiene gli hub ("illegal immigration", "Trump", "deportation", "US news"...) più molte keyword *satellite*, che nella rete diluita hanno pochissimi archi (grado mediano 2) e tutti verso gli hub, perché la soglia di peso 5 ha tolto i loro legami specifici. Non è un tema: è il nucleo generico del discorso su ICE più le keyword senza un'associazione propria. Al crescere di γ i satelliti si staccano in piccoli gruppi e il nucleo scende sotto un quarto della rete. In §5 va letto come tale.

**Scelta: γ = 3 per tutte le reti di keyword, γ = 2 per gli autori.**

- *Keyword*: a γ = 3 le reti diluite sono vicine alla loro stabilità massima: NYT 0,97 (0,98 a γ = 2, dove però il nucleo ha ancora il 36% dei nodi), Guardian 0,95 (0,98 a γ = 4, dove solo il 69% dei nodi è in comunità leggibili); il nucleo scende al 22-23% e le comunità leggibili sono 21 (NYT) e 11 (Guardian). Prezzo: nel Guardian il 25% dei nodi finisce in gruppi di 1-4 keyword (i satelliti staccati). Le reti di controllo a γ = 3 sono stabili (0,84-0,90) e senza nucleo. Stesso γ per i due giornali e per le due reti, così le partizioni sono confrontabili.
- *Autori* (scelta rivista il 2026-10-05, quando la rete è passata da 117 a 358 autori con il 2025+): a γ = 1, il valore standard scelto in precedenza, la partizione ora è instabile (AMI 0,75). A γ = 2 è stabile (AMI 0,86 nel backbone, 0,91 nel controllo), con 17 comunità di cui 16 con almeno 5 autori e la maggiore al 10%. γ = 3 è appena più stabile (0,90 e 0,93) ma spezza filoni coerenti in gruppi più piccoli e lascia da solo l'autore più prolifico (Hamed Aleaziz); si tiene γ = 2, lo stesso per le due reti per confrontarle.

### 4.3 Partizione rappresentativa e confronto con reti casuali

**Partizione rappresentativa.** Con la risoluzione scelta i seed danno partizioni molto simili ma non identiche. Si tiene quella con l'AMI medio più alto rispetto a tutte le altre: la partizione "più centrale" tra i `N_SEEDS` seed. Le comunità sono rinumerate per dimensione (0 = la più grande).

**Confronto con reti casuali.** Una modularità positiva non basta: anche una rete casuale, ottimizzata da Leiden, ha comunità con modularità > 0. Per ogni rete si generano `N_RANDOM` reti casuali con **lo stesso numero di archi per nodo** (scambio casuale degli estremi degli archi, *double edge swap*) e gli stessi pesi riassegnati a caso agli archi, e si esegue Leiden con la stessa risoluzione.

**Riproducibilità.** `largest_component` riordina nodi e archi alfabeticamente: `subgraph()` di networkx itera su un set, quindi senza riordinare l'ordine dei vertici passato a Leiden (e alle reti casuali) cambiava a ogni avvio del kernel e i risultati cambiavano anche a seed fisso. Verificato: due esecuzioni con `PYTHONHASHSEED` diversi danno output identici. Se la modularità della rete vera è molto sopra quella delle reti casuali (z-score), le comunità riflettono una struttura reale e non sono un effetto della distribuzione dei gradi.

In [ ]:
# Fissiamo i valori ottimali di γ per ogni combinazione di rete e qualità, scelti in base a modularità, AMI tra seed e dimensioni delle comunità.
CHOSEN_RESOLUTION = {
    ("nyt", "keyword, diluita"): 3.0,
    ("nyt", "keyword, controllo"): 3.0,
    ("guardian", "keyword, diluita"): 3.0,
    ("guardian", "keyword, controllo"): 3.0,
    ("nyt", "autori, backbone α=0,05"): 2.0,
    ("nyt", "autori, controllo α=0,01"): 2.0,
}
N_RANDOM = 20

# Ritorna la partizione con l'AMI medio più alto verso le altre, rinumerando le comunità per dimensione decrescente.
# In questo modo troviamo una partizione rappresentativa tra i diversi seed, che possiamo usare per confrontare le reti tra loro.
def representative_partition(runs: list, nodes: list[str]) -> pd.Series:
    """Partizione con l'AMI medio più alto verso le altre; comunità rinumerate per dimensione decrescente."""
    memberships = [np.array(p.membership) for p in runs]
    centrality = [np.mean([adjusted_mutual_info_score(m, other) for other in memberships]) for m in memberships]
    best = pd.Series(memberships[int(np.argmax(centrality))], index=nodes)
    order = {old: new for new, old in enumerate(best.value_counts().index)}
    return best.map(order).rename("community")

# Ritorna un grafo con gli stessi gradi e pesi del grafo originale, ma con i pesi assegnati casualmente agli archi.
# Serve a creare una rete casuale con stesse proprietà matematiche del grafo originale ma senza significato semantico.
def randomized(graph: nx.Graph, weight: str, seed: int) -> nx.Graph:
    """Stessi gradi (double edge swap) e stessi pesi, assegnati a caso agli archi."""
    rng = np.random.default_rng(seed)
    rand = nx.Graph(graph.edges())
    nx.double_edge_swap(rand, nswap=10 * rand.number_of_edges(), max_tries=1000 * rand.number_of_edges(), seed=seed)
    weights = rng.permutation([d[weight] for _, _, d in graph.edges(data=True)])
    for (u, v), w in zip(rand.edges(), weights):
        rand[u][v][weight] = w
    return rand

# Prendo la rete reale, estraggo la partizione rappresentativa, calcolo la modularità e:
# Genero 20 reti casuali, applico Leiden in ciascuna, calcolo la modularità casuale e lo Z-score
# (Confronta la modularità della tua rete reale con la media delle 20 reti finte. Se lo Z-score è molto alto (es. > 2 o 3), significa 
# che la struttura a cluster che hai trovato non è dovuta alla struttura matematica del grafo, ma ha un significato semantico/sociale)
partitions = {}
rows = []
for key, (graph, weight) in NETWORKS.items():
    resolution = CHOSEN_RESOLUTION[key]
    _, nodes = to_igraph(graph, weight)
    runs = resolution_runs[(*key, resolution)]
    partitions[key] = representative_partition(runs, nodes)
    observed = runs[int(np.argmax([np.mean([adjusted_mutual_info_score(r.membership, o.membership) for o in runs]) for r in runs]))].modularity

    null = []
    for seed in range(N_RANDOM):
        g_rand, _ = to_igraph(randomized(graph, weight, seed), weight)
        null.append(leiden(g_rand, "modularità", resolution, seed).modularity)
    sizes = partitions[key].value_counts()
    rows.append({
        "giornale": key[0], "rete": key[1], "γ": resolution, "nodi": len(nodes),
        "comunità": len(sizes), f"comunità_≥{MIN_COMMUNITY_SIZE}": int((sizes >= MIN_COMMUNITY_SIZE).sum()),
        "modularità": round(observed, 3),
        "modularità_casuale": f"{np.mean(null):.3f} ± {np.std(null):.3f}",
        "z": round((observed - np.mean(null)) / np.std(null), 1),
    })
    log.info("%s / %s: partizione rappresentativa e %d reti casuali fatte", *key, N_RANDOM)
pd.DataFrame(rows).set_index(["giornale", "rete"])

**Come leggere la tabella.** Una riga per rete, con la risoluzione scelta. `modularità` = modularità della partizione rappresentativa; `modularità_casuale` = media ± deviazione standard sulle `N_RANDOM` reti casuali con gli stessi gradi e gli stessi pesi; `z` = di quante deviazioni standard la rete vera supera quelle casuali. Un valore sopra 3 vuol dire che una modularità così alta non si ottiene per caso con quella distribuzione dei gradi.

**Risultato.** Tutte e sei le reti superano di molto le reti casuali (z tra 16 e 120): le comunità non sono un effetto dei gradi dei nodi o della distribuzione dei pesi. Il valore assoluto della modularità delle reti di keyword è basso (0,08-0,18): i temi si sovrappongono molto (quasi ogni articolo su ICE tocca anche "deportation" o "Trump"), quindi i confini tra comunità sono sfumati. Va detto nel report: la struttura c'è ed è significativa, ma non è fatta di blocchi separati.

Partizioni rappresentative scelte (in `partitions`): keyword NYT 22 comunità di almeno 5 nodi (31 in tutto), Guardian 11 (29); controllo 32 (42) e 21 (29); autori 16 (17 in tutto), controllo autori 18 (18). Le comunità vengono lette in §5 (keyword) e §6 (autori).

## 5. Comunità delle keyword (NYT e Guardian)

Si leggono le partizioni rappresentative della §4 sulle **reti diluite** (γ = 3), solo le comunità con almeno `MIN_COMMUNITY_SIZE = 5` keyword. Quattro domande, un grafico ciascuna (più uno per giornale):

1. **Che cosa contiene ogni comunità** (§5.1): dimensione, keyword principali, categorie, quanto è chiusa su se stessa.
2. **Quando conta** (§5.2): in quale periodo gli articoli usano le sue keyword.
3. **È robusta?** (§5.3): le stesse keyword stanno insieme anche nella rete di controllo, costruita senza la soglia di peso 5?
4. **NYT e Guardian** (§5.4): quali temi hanno entrambi e con che peso, allineati a mano perché le tassonomie sono diverse.

**Etichette.** Ogni comunità leggibile ha un'etichetta scritta a mano leggendo le sue keyword (`COMMUNITY_LABELS`). L'etichetta è legata a una keyword "firma" e non al numero della comunità, che cambierebbe con la partizione: se una firma esce dalla rete, finisce nella stessa comunità di un'altra, o resta una comunità leggibile senza etichetta, la cella si ferma con un errore. Le etichette sono un'interpretazione: vanno lette insieme alle keyword nei grafici.

**Nucleo.** La comunità che contiene la keyword più frequente (`illegal immigration` per il NYT, `US news` per il Guardian) è il nucleo generico visto in §4.2: hub più keyword satellite. Nei grafici è grigio e non va letto come un tema.

I grafici sono salvati anche in `docs/figures/community_*.png`. **Lingua**: i grafici sono in inglese (per il report), comprese le etichette delle comunità; i testi del notebook restano in italiano e citano le etichette con il nome inglese.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

FIGURES_DIR = BASE_DIR / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# palette categoriale (ordine fisso), grigio per il nucleo generico, inchiostri per testo e assi
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
PAPER_COLORS = {"nyt": SERIES[0], "guardian": SERIES[1]}
PAPER_NAMES = {"nyt": "NYT", "guardian": "Guardian"}
CORE_COLOR = "#b9b7b0"
INK, INK_SECONDARY, GRID = "#0b0b0b", "#52514e", "#e4e3df"
CATEGORY_COLORS = dict(zip(["subject", "glocations", "persons", "organizations", "creative_works"], SERIES))
CATEGORY_NAMES = {"subject": "subject", "glocations": "place", "persons": "person",
                  "organizations": "organization", "creative_works": "creative work"}

plt.rcParams.update({
    "font.size": 9, "axes.edgecolor": INK_SECONDARY, "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_SECONDARY, "ytick.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.dpi": 150, "savefig.bbox": "tight",
})

# Etichette scritte a mano leggendo le keyword di ogni comunità (≥ 5 nodi) della rete diluita.
# Chiave = una keyword "firma" che deve stare nella comunità: le etichette non dipendono dal numero
# della comunità, che cambierebbe se cambiasse la partizione.
COMMUNITY_LABELS = {
    "nyt": {
        "illegal immigration": "Generic core (immigration, detention, deportation)",
        "trump, donald j": "Trump administration and federal government",
        "family separation policy (us immigration)": "Border, asylum and family separation",
        "habeas corpus": "Courts and lawsuits against the government",
        "delaney hall (newark, nj)": "Delaney Hall (Newark)",
        "adams, eric l": "New York City: Adams and Rikers",
        "national guard": "Protests and National Guard (Los Angeles, Portland)",
        "democratic party": "Parties and elections",
        "alien enemies act (1798)": "Deportations to El Salvador, Alien Enemies Act",
        "police brutality, misconduct and shootings": "Shootings and deaths during operations",
        "foreign students (in us)": "Foreign students and visas",
        "shutdowns (institutional)": "Shutdown and airport security",
        "obama, barack": "Obama years: illegal immigrants and crime",
        "minneapolis (minn)": "Minneapolis (2026)",
        "hochul, kathleen c": "New York State: legislation (Hochul, Mamdani)",
        "foreign workers": "Migrant labor",
        "columbia university": "Pro-Palestinian campus protests",
        "chicago (ill)": "Chicago and federal-state relations",
        "social media": "Technology and social media",
        "united states international relations": "International relations and human rights",
        "terrorism": "Terrorism and federal policing in Washington",
        "guantanamo bay naval base (cuba)": "Military and Guantánamo",
    },
    "guardian": {
        "us-news/us-news": "Generic core (US news, US immigration)",
        "world/world": "Americas and migration (El Salvador, Venezuela)",
        "us-news/minneapolis": "Minneapolis (2026)",
        "law/law": "Law, civil liberties and protests",
        "us-news/trump-administration": "Trump administration",
        "us-news/california": "California and Los Angeles",
        "us-news/mahmoud-khalil": "Pro-Palestinian campus protests",
        "culture/television": "Culture and television",
        "us-news/texas": "Texas and gun crime",
        "us-news/chicago": "Chicago and the military",
        "us-news/democrats": "Congress and parties",
    },
}
CORE_KEYWORD = {"nyt": "illegal immigration", "guardian": "us-news/us-news"}
MAIN_KEY = {paper: (paper, "keyword, diluita") for paper in PAPERS}
CONTROL_KEY = {paper: (paper, "keyword, controllo") for paper in PAPERS}


def readable_communities(key) -> pd.Series:
    """community -> lista dei nodi, solo comunità con almeno MIN_COMMUNITY_SIZE nodi."""
    groups = partitions[key].groupby(partitions[key]).apply(lambda s: sorted(s.index))
    return groups[groups.apply(len) >= MIN_COMMUNITY_SIZE]


def community_labels(paper: str) -> dict[int, str]:
    """community -> etichetta manuale, ricavata dalla keyword firma; errore se una firma manca o è doppia."""
    part = partitions[MAIN_KEY[paper]]
    labels = {}
    for signature, label in COMMUNITY_LABELS[paper].items():
        if signature not in part.index:
            raise KeyError(f"{paper}: keyword firma '{signature}' non è nella rete")
        community = int(part[signature])
        if community in labels:
            raise ValueError(f"{paper}: '{signature}' e un'altra firma sono nella stessa comunità {community}")
        labels[community] = label
    missing = set(readable_communities(MAIN_KEY[paper]).index) - set(labels)
    if missing:
        raise ValueError(f"{paper}: comunità leggibili senza etichetta: {sorted(missing)}")
    return labels


def shorten(text: str, width: int = 22) -> str:
    return text if len(text) <= width else text[:width - 1] + "…"


def community_summary(paper: str) -> pd.DataFrame:
    """Una riga per comunità leggibile della rete diluita: dimensione, keyword principali, categorie, legami interni."""
    graph, _ = NETWORKS[MAIN_KEY[paper]]
    part = partitions[MAIN_KEY[paper]]
    labels = community_labels(paper)
    core = int(part[CORE_KEYWORD[paper]])
    rows = []
    for community, nodes in readable_communities(MAIN_KEY[paper]).items():
        members = set(nodes)
        strength = sum(d["assoc"] for n in nodes for _, _, d in graph.edges(n, data=True))
        internal = sum(d["assoc"] for n in nodes for _, v, d in graph.edges(n, data=True) if v in members)
        by_count = sorted(nodes, key=lambda n: (-graph.nodes[n]["count"], n))
        categories = pd.Series([graph.nodes[n]["category"] for n in nodes]).value_counts(normalize=True)
        rows.append({
            "community": community, "etichetta": labels[community], "nucleo": community == core,
            "keyword": len(nodes),
            "keyword_principali": " · ".join(shorten(node_label(graph, n)) for n in by_count[:3]),
            "legami_interni_pct": round(internal / strength * 100, 1),
            **{f"cat_{c}": categories.get(c, 0.0) for c in CATEGORY_COLORS},
        })
    return pd.DataFrame(rows).sort_values("keyword", ascending=False).reset_index(drop=True)


SUMMARIES = {paper: community_summary(paper) for paper in PAPERS}
for paper, summary in SUMMARIES.items():
    covered = summary["keyword"].sum() / len(partitions[MAIN_KEY[paper]]) * 100
    log.info("%s: %d comunità leggibili, %.1f%% dei nodi; nucleo = %d keyword",
             paper, len(summary), covered, summary.loc[summary["nucleo"], "keyword"].iloc[0])

**Come leggere il log.** Per giornale: quante comunità hanno almeno 5 keyword, che quota dei nodi della rete diluita coprono, quante keyword ha il nucleo.

### 5.1 Profilo delle comunità

Un grafico per giornale, una riga per comunità leggibile, ordinate per dimensione. Tre pannelli:

- **dimensione**: numero di keyword; accanto, le 3 keyword della comunità presenti in più articoli (troncate se lunghe);
- **composizione per categoria**: quota di temi, luoghi, persone, organizzazioni. Per il Guardian è il `keywordType` del tag rimappato sulle categorie NYT (notebook 08);
- **legami interni**: quota dell'association strength dei nodi della comunità che resta tra nodi della stessa comunità. Alta = la comunità è un blocco chiuso (una vicenda raccontata sempre con le stesse keyword); bassa = le sue keyword sono legate anche a molte altre comunità (temi trasversali).

In [ ]:
def plot_community_profile(paper: str, out_path: Path) -> None:
    """Tre pannelli con le comunità sulle righe: dimensione, composizione per categoria, legami interni."""
    summary = SUMMARIES[paper]
    y = np.arange(len(summary))[::-1]
    fig, (ax_size, ax_cat, ax_int) = plt.subplots(
        1, 3, figsize=(13, 0.34 * len(summary) + 1.6), sharey=True,
        gridspec_kw={"width_ratios": [2.2, 1, 1], "wspace": 0.08},
    )

    colors = [CORE_COLOR if core else PAPER_COLORS[paper] for core in summary["nucleo"]]
    ax_size.barh(y, summary["keyword"], color=colors, height=0.7)
    for yi, (n, top) in zip(y, summary[["keyword", "keyword_principali"]].itertuples(index=False)):
        ax_size.text(n + 1, yi, f"{n}   {top}", va="center", fontsize=7, color=INK_SECONDARY)
    ax_size.set_xlim(0, summary["keyword"].max() * 3.5)
    ax_size.set_xlabel("keywords in the community  (text: the 3 most frequent in articles)")
    ax_size.set_yticks(y)
    ax_size.set_yticklabels(summary["etichetta"])

    left = np.zeros(len(summary))
    for category, color in CATEGORY_COLORS.items():
        share = summary[f"cat_{category}"].to_numpy() * 100
        if share.sum() == 0:
            continue
        ax_cat.barh(y, share, left=left, color=color, height=0.7, edgecolor="white", linewidth=1,
                    label=CATEGORY_NAMES[category])
        left += share
    ax_cat.set_xlim(0, 100)
    ax_cat.set_xlabel("composition by category (%)")
    ax_cat.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=5, frameon=False, fontsize=7,
                  handlelength=1, columnspacing=0.8)

    ax_int.barh(y, summary["legami_interni_pct"], color=colors, height=0.7)
    for yi, v in zip(y, summary["legami_interni_pct"]):
        ax_int.text(v + 1, yi, f"{v:.0f}%", va="center", fontsize=7, color=INK_SECONDARY)
    ax_int.set_xlim(0, 100)
    ax_int.set_xlabel("internal links (% of strength, assoc)")

    for ax in (ax_size, ax_cat, ax_int):
        ax.grid(axis="x", color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)
        ax.tick_params(axis="y", length=0)
    n_nodes = len(partitions[MAIN_KEY[paper]])
    fig.suptitle(f"{PAPER_NAMES[paper]} — keyword communities of the pruned network (γ = 3): "
                 f"{len(summary)} communities with at least {MIN_COMMUNITY_SIZE} keywords, "
                 f"{summary['keyword'].sum()} of {n_nodes} nodes", x=0.02, ha="left", fontsize=11, color=INK)
    fig.savefig(out_path)
    plt.show()


for paper in PAPERS:
    plot_community_profile(paper, FIGURES_DIR / f"community_profile_{paper}.png")

**Come leggere i grafici.** Ogni riga è una comunità: l'etichetta a sinistra, la barra lunga quante keyword contiene, i colori che dicono di che tipo sono le keyword, a destra quanto la comunità è chiusa. Il nucleo è in grigio.

**Risultato.**

- **NYT**: 22 comunità leggibili, 341 nodi su 366 (93%). Oltre al nucleo (79 keyword) c'è una seconda comunità grande, *Trump administration and federal government* (51): come il nucleo ha pochi legami interni (26%, nucleo 35%), cioè raccoglie keyword trasversali (Trump, DHS, "united states politics and government", nomine). Le altre 20 comunità hanno 5-20 keyword e quasi sempre più del 60% di legami interni: sono **vicende** (Delaney Hall a Newark, Minneapolis 2026, studenti stranieri e visti, El Salvador e Alien Enemies Act, sparatorie durante le operazioni) o **filoni** (tribunali, partiti ed elezioni, lavoro dei migranti, anni di Obama).
- La composizione distingue i due tipi: le vicende hanno molte persone e luoghi (Minneapolis: 90% persone e luoghi; Delaney Hall: 64% persone e organizzazioni), i filoni sono quasi solo temi (lavoro dei migranti: 100%).
- **Guardian**: 11 comunità leggibili, 128 nodi su 172 (74%); il resto sono gruppi di 1-4 tag (§4.2). Stessa struttura: nucleo (40 tag, 36% di legami interni) e *Trump administration* trasversale (20%), poi vicende e filoni. *Culture and television* (100%) e *Congress and parties* (85%, solo organizzazioni) sono gruppi di tag annidati o quasi sinonimi (§2): ovvi, non scelte editoriali.
- I legami interni bassi anticipano la §5.3: le comunità trasversali sono quelle che nella rete di controllo si sciolgono.

### 5.2 Presenza delle comunità nel tempo

Per ogni comunità e periodo: quota degli articoli del periodo che hanno **almeno una** keyword della comunità. Si usano tutti gli articoli del corpus (2.093 NYT, 1.738 Guardian), con le keyword estratte come nei notebook 02 e 08.

Il nucleo è escluso dalla mappa di colore (è in quasi tutti gli articoli e schiaccerebbe le altre righe); i suoi valori sono nel titolo. Limite: basta una keyword, quindi una comunità con una keyword molto generica (es. "united states politics and government" nella comunità Trump, `World news` nelle Americhe del Guardian) risulta presente in molti articoli anche fuori dal suo tema.

In [ ]:
GUARDIAN_ICE_TAG = "us-news/ice-us-immigration-and-customs-enforcement"
PERIOD_BOUNDS = {"2003-2016": (2003, 2016), "2017-2020": (2017, 2020), "2021-2024": (2021, 2024), "2025-presente": (2025, 9999)}


def period_of(year: int) -> str:
    return next(label for label, (start, end) in PERIOD_BOUNDS.items() if start <= year <= end)


def load_article_keywords(paper: str) -> pd.DataFrame:
    """Una riga per articolo del corpus: periodo e insieme delle keyword (stessi id dei nodi della rete)."""
    if paper == "nyt":
        df = pd.read_parquet(PAPERS[paper]["articles"], columns=["pub_date", "keywords_json"])
        df["keywords"] = [set(extract_keywords(json.loads(k))) for k in df["keywords_json"]]
    else:
        df = pd.read_parquet(PAPERS[paper]["articles"], columns=["pub_date", "keyword_tags_json", "is_ice_relevant"])
        df = df[df["is_ice_relevant"]].copy()
        df["keywords"] = [set(json.loads(k)) - {GUARDIAN_ICE_TAG} for k in df["keyword_tags_json"]]
    df["period"] = df["pub_date"].dt.year.map(period_of)
    return df[["period", "keywords"]]


def community_presence(paper: str) -> pd.DataFrame:
    """% degli articoli di ogni periodo che hanno almeno una keyword della comunità (righe = comunità leggibili)."""
    articles = load_article_keywords(paper)
    groups = readable_communities(MAIN_KEY[paper])
    labels = community_labels(paper)
    presence = {}
    for community in SUMMARIES[paper]["community"]:
        members = set(groups[community])
        hit = articles["keywords"].apply(lambda kws: not kws.isdisjoint(members))
        presence[labels[community]] = hit.groupby(articles["period"]).mean() * 100
    table = pd.DataFrame(presence).T[PERIOD_LABELS]
    table.columns = [f"{p.replace('presente', 'present')}\n({(articles['period'] == p).sum()} articles)" for p in PERIOD_LABELS]
    return table


def plot_presence(paper: str, out_path: Path) -> pd.DataFrame:
    table = community_presence(paper)
    core_label = SUMMARIES[paper].loc[SUMMARIES[paper]["nucleo"], "etichetta"].iloc[0]
    # il nucleo è in quasi tutti gli articoli: tenerlo nella scala schiaccerebbe le altre righe
    shown = table.drop(index=core_label)
    vmax = shown.to_numpy().max()

    fig, ax = plt.subplots(figsize=(7.5, 0.3 * len(shown) + 1.6))
    im = ax.imshow(shown.to_numpy(), cmap="Blues", vmin=0, vmax=vmax, aspect="auto")
    for (i, j), v in np.ndenumerate(shown.to_numpy()):
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                color="white" if v > vmax * 0.6 else INK)
    ax.set_xticks(range(len(shown.columns)))
    ax.set_xticklabels(shown.columns, fontsize=8)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(shown)))
    ax.set_yticklabels(shown.index)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("% of the period's articles with at least one community keyword", fontsize=7)
    cbar.outline.set_visible(False)
    core_values = ", ".join(f"{v:.0f}%" for v in table.loc[core_label])
    ax.set_title(f"{PAPER_NAMES[paper]} — community presence over time  (generic core, not shown: {core_values})",
                 loc="left", fontsize=10, color=INK, pad=28)
    fig.savefig(out_path)
    plt.show()
    return table


PRESENCE = {paper: plot_presence(paper, FIGURES_DIR / f"community_presence_{paper}.png") for paper in PAPERS}

**Come leggere i grafici.** Righe = comunità (stesso ordine del §5.1), colonne = periodi con il numero di articoli. Il numero nella cella è la % degli articoli del periodo che usa almeno una keyword della comunità; più scuro = più presente. Le righe si leggono da sinistra a destra: dicono quando un tema entra o esce dalla copertura.

**Risultato.**

- **NYT**: le comunità datate si riconoscono subito. *Obama years* è nel 58% degli articoli 2003-2016 e poi scompare (1-2%); *Border, asylum and family separation* sale dal 23% al 39% (2017-2020) e al 69% nel 2021-2024 (crisi al confine sotto Biden), poi scende al 18%; *Shutdown and airports* è al 25% nel 2021-2024. Nel 2025-presente emergono le città: *Protests and National Guard* (27%), *Minneapolis* (22%, 0% prima), *Chicago* (17%), *Shootings and deaths during operations* (13%). *Trump administration* cresce in modo continuo (33% → 84%).
- **Guardian**: *Minneapolis* (22% nel 2025+) e *Pro-Palestinian campus protests* compaiono solo nel 2025; *Americas and migration* scende dal 61% al 33% (la copertura passa dal confine e dall'America Latina alle città statunitensi, come nella NER del notebook 06); *Trump administration* segue i mandati (11% → 49% → 30% → 70%).
- Spostamento comune ai due giornali: dal confine e dall'America Latina alle operazioni nelle città degli Stati Uniti nel 2025+.

### 5.3 Rete diluita contro rete di controllo

Questa sezione serve come controllo di robustezza, serve a capire quali comunità sono un risultato vero e quali dipendono da una scelta arbitraria precedente. Uno dei problemi potrebbe essere la soglia di peso 5 applicata sugli archi. Utilizziamo dunque una rete di controllo in cui al posto degli archi deboli filtriamo le keyqord rare (che compaiono in meno di 5 articoli) tenendo dunque solo gli archi presenti tra le keyword rimaste. La rete ottenuta è di 434 nodi (keyword) e 12.890 archi. Le keyword saranno più o meno le stesse ma i collegamenti saranno diversi. Facciamo poi girare lo stesso Leiden (stessi parametri) su entrambe le reti e confrontiamo le partizioni: se una comunità esce uguale in entrambe vuol dire che è robusta (perchè il gruppo di keyword sta insieme anche tolto il rumore), altrimenti esiste solamente grazie alla soglia applicata. Per la selezione applichiamo 2 misure:

- **AMI** tra le due partizioni, con e senza le keyword del nucleo (il nucleo esiste solo nella diluita: le sue keyword nel controllo si distribuiscono tra molte comunità, e questo da solo abbassa l'AMI);
- per ogni comunità diluita, **dove finiscono** le sue keyword nel controllo (mappa di colore). Una comunità è robusta se gran parte delle sue keyword resta insieme in un'unica comunità di controllo.

In [ ]:
def main_vs_control(paper: str) -> tuple[pd.DataFrame, float, float]:
    """Righe = comunità leggibili della diluita, colonne = comunità del controllo: % delle keyword della riga
    che finiscono in quella colonna. Più l'AMI sulle keyword presenti in entrambe le reti, con e senza nucleo."""
    main, control = partitions[MAIN_KEY[paper]], partitions[CONTROL_KEY[paper]]
    shared = main.index.intersection(control.index)
    ami_all = adjusted_mutual_info_score(main[shared], control[shared])
    core = main[CORE_KEYWORD[paper]]
    no_core = [n for n in shared if main[n] != core]
    ami_no_core = adjusted_mutual_info_score(main[no_core], control[no_core])

    labels = community_labels(paper)
    rows = [c for c in SUMMARIES[paper]["community"]]
    sub = pd.DataFrame({"main": main[shared], "control": control[shared]})
    sub = sub[sub["main"].isin(rows)]
    table = pd.crosstab(sub["main"], sub["control"], normalize="index") * 100
    table = table.loc[rows]
    # colonne in ordine di "prima riga che le usa di più", per avvicinare la diagonale
    order = sorted(table.columns, key=lambda c: (int(np.argmax(table[c].to_numpy() == table[c].max())), -table[c].max()))
    table = table[order]
    table.index = [labels[c] for c in rows]
    return table, ami_all, ami_no_core


def plot_main_vs_control(paper: str, out_path: Path) -> pd.DataFrame:
    table, ami_all, ami_no_core = main_vs_control(paper)
    fig, ax = plt.subplots(figsize=(0.22 * table.shape[1] + 4.5, 0.3 * table.shape[0] + 1.6))
    ax.imshow(table.to_numpy(), cmap="Blues", vmin=0, vmax=100, aspect="auto")
    for (i, j), v in np.ndenumerate(table.to_numpy()):
        if v >= 50:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6, color="white")
    ax.set_yticks(range(len(table)))
    ax.set_yticklabels(table.index)
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels(table.columns, fontsize=6)
    ax.set_xlabel("control network community (id)")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f"{PAPER_NAMES[paper]} — where the keywords of each pruned-network community end up in the control network\n"
                 f"AMI on shared keywords {ami_all:.2f}, without the generic core {ami_no_core:.2f}  "
                 f"(colour = % of the row's keywords, number shown if ≥ 50%)",
                 loc="left", fontsize=10, color=INK)
    fig.savefig(out_path)
    plt.show()
    best = table.max(axis=1).rename("quota_max_pct")
    log.info("%s: AMI diluita vs controllo %.2f (senza nucleo %.2f); comunità con ≥ 50%% delle keyword in un'unica comunità di controllo: %d su %d",
             paper, ami_all, ami_no_core, (best >= 50).sum(), len(best))
    return best


CONTROL_AGREEMENT = {paper: plot_main_vs_control(paper, FIGURES_DIR / f"community_vs_control_{paper}.png") for paper in PAPERS}

**Come leggere i grafici.** Righe = comunità della diluita, colonne = comunità della rete di controllo (indicate col numero, le etichette esistono solo per la diluita). Il colore è la % delle keyword della riga che finisce in quella colonna; il numero è scritto se supera il 50%. Una riga con un'unica cella scura = la comunità esiste uguale anche senza la soglia di peso; una riga chiara e sparsa = le sue keyword nel controllo si dividono.

**Risultato.**

- **AMI moderato**: 0,38 NYT e 0,39 Guardian, 0,50 e 0,52 senza il nucleo. Le due partizioni nel complesso sono diverse, perché nel controllo non c'è il nucleo e le keyword generiche si distribuiscono ovunque.
- **Le vicende sono robuste**: 17 comunità NYT su 22 e 7 Guardian su 11 hanno almeno metà delle keyword in un'unica comunità di controllo. Quasi identiche: proteste pro-Palestina (100% NYT, 88% Guardian), El Salvador e Guardia nazionale (92%), anni di Obama e shutdown (90%), tecnologia (86%).
- **Si sciolgono le comunità trasversali**, le stesse con pochi legami interni in §5.1: nucleo (8% NYT, 15% Guardian), *Trump administration* (16% NYT, 22% Guardian), *Courts and lawsuits* (19%), *International relations* (29%), *Parties and elections* (38%), *Law, civil liberties and protests* e *Americas and migration* nel Guardian (22%, 30%).
- **Conseguenza per il report**: si possono citare come risultati le comunità-vicenda e i filoni stabili; le comunità trasversali dicono solo che quelle keyword sono legate a tutto, e la loro composizione esatta dipende da come si tolgono gli archi deboli.

### 5.4 NYT contro Guardian

Le tassonomie non sono allineate (`illegal immigration` contro `US immigration`), quindi le comunità non si confrontano nodo per nodo. Le comunità leggibili sono state raggruppate a mano in **temi** (`SHARED_THEMES`): stesso argomento nei due giornali, anche se con keyword diverse. Per ogni tema la dimensione è la quota dei nodi della rete diluita del giornale (le reti hanno dimensioni diverse: 366 e 172 nodi). Ogni comunità leggibile deve stare in un tema, altrimenti la cella si ferma con un errore.

In [ ]:
# Temi allineati a mano tra i due giornali: nome del tema -> etichetta della comunità in ciascun giornale
# (None = nessuna comunità leggibile corrispondente). Le tassonomie sono diverse, l'allineamento è per significato.
SHARED_THEMES = {
    "Generic core": ("Generic core (immigration, detention, deportation)", "Generic core (US news, US immigration)"),
    "Trump administration": ("Trump administration and federal government", "Trump administration"),
    "Latin America, El Salvador": ("Deportations to El Salvador, Alien Enemies Act", "Americas and migration (El Salvador, Venezuela)"),
    "Border and asylum": ("Border, asylum and family separation", None),
    "Courts and rights": ("Courts and lawsuits against the government", "Law, civil liberties and protests"),
    "Parties, Congress, elections": ("Parties and elections", "Congress and parties"),
    "Minneapolis (2026)": ("Minneapolis (2026)", "Minneapolis (2026)"),
    "Los Angeles, California": ("Protests and National Guard (Los Angeles, Portland)", "California and Los Angeles"),
    "Chicago": ("Chicago and federal-state relations", "Chicago and the military"),
    "Pro-Palestinian campus protests": ("Pro-Palestinian campus protests", "Pro-Palestinian campus protests"),
    "Shootings, crime": ("Shootings and deaths during operations", "Texas and gun crime"),
    "New York (city and state)": ("New York City: Adams and Rikers", None),
    "Newark, Delaney Hall": ("Delaney Hall (Newark)", None),
    "Foreign students and visas": ("Foreign students and visas", None),
    "Migrant labor": ("Migrant labor", None),
    "Shutdown and airports": ("Shutdown and airport security", None),
    "Obama years": ("Obama years: illegal immigrants and crime", None),
    "Technology and social media": ("Technology and social media", None),
    "International relations": ("International relations and human rights", None),
    "Terrorism, Washington": ("Terrorism and federal policing in Washington", None),
    "Military and Guantánamo": ("Military and Guantánamo", None),
    "Culture and television": (None, "Culture and television"),
}
# "New York State: legislation" va con New York City nello stesso tema
EXTRA_NYT = {"New York (city and state)": ["New York State: legislation (Hochul, Mamdani)"]}


def theme_sizes() -> pd.DataFrame:
    """% dei nodi della rete diluita in ciascun tema, per giornale (NaN = tema senza comunità)."""
    used = {paper: set() for paper in PAPERS}
    rows = []
    for theme, labels in SHARED_THEMES.items():
        row = {"tema": theme}
        for paper, label in zip(["nyt", "guardian"], labels):
            summary = SUMMARIES[paper].set_index("etichetta")
            wanted = [label] + (EXTRA_NYT.get(theme, []) if paper == "nyt" else []) if label else []
            for w in wanted:
                if w not in summary.index:
                    raise KeyError(f"{paper}: etichetta '{w}' non trovata")
            used[paper].update(wanted)
            n_nodes = len(partitions[MAIN_KEY[paper]])
            row[paper] = summary.loc[wanted, "keyword"].sum() / n_nodes * 100 if wanted else np.nan
        rows.append(row)
    for paper in PAPERS:
        unused = set(SUMMARIES[paper]["etichetta"]) - used[paper]
        if unused:
            raise ValueError(f"{paper}: comunità non assegnate a un tema: {sorted(unused)}")
    return pd.DataFrame(rows).set_index("tema")


def plot_shared_themes(out_path: Path) -> pd.DataFrame:
    sizes = theme_sizes()
    both = sizes.dropna().index
    order = list(both.drop("Generic core")) + [t for t in sizes.index if t not in both]
    order = ["Generic core"] + sorted(order, key=lambda t: (t not in both, -np.nanmax(sizes.loc[t].to_numpy())))
    sizes = sizes.loc[order]
    y = np.arange(len(sizes))[::-1]

    fig, ax = plt.subplots(figsize=(8, 0.3 * len(sizes) + 1.4))
    for yi, (nyt, guardian) in zip(y, sizes[["nyt", "guardian"]].itertuples(index=False)):
        if not np.isnan(nyt) and not np.isnan(guardian):
            ax.plot([nyt, guardian], [yi, yi], color=GRID, linewidth=2, zorder=1)
    for paper in PAPERS:
        values = sizes[paper].to_numpy()
        ax.scatter(values, y, s=42, color=PAPER_COLORS[paper], edgecolor="white", linewidth=1.5,
                   zorder=2, label=PAPER_NAMES[paper])
    ax.axhline(y[len(both) - 1] - 0.5, color=INK_SECONDARY, linewidth=0.6, linestyle=":")
    ax.text(0.99, y[len(both) - 1] - 0.5, "above: in both · below: in one newspaper only ",
            transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=7, color=INK_SECONDARY)
    ax.set_xscale("log")
    ax.set_xticks([1, 2, 5, 10, 20])
    ax.set_xticklabels(["1%", "2%", "5%", "10%", "20%"])
    ax.set_yticks(y)
    ax.set_yticklabels(sizes.index)
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_xlabel("% of the newspaper's pruned-network nodes (log scale)")
    ax.legend(loc="lower right", frameon=False)
    ax.set_title("Keyword community themes, NYT vs Guardian (aligned by hand)", loc="left", fontsize=10, color=INK)
    fig.savefig(out_path)
    plt.show()
    return sizes


THEME_SIZES = plot_shared_themes(FIGURES_DIR / "community_themes_nyt_vs_guardian.png")

**Come leggere il grafico.** Una riga per tema; punto blu = NYT, arancione = Guardian, posizione = % dei nodi della rete del giornale (scala logaritmica: ogni tacca raddoppia o quintuplica). Sopra la linea tratteggiata i temi che hanno una comunità in entrambi i giornali, unite da una barra grigia quanto più diversa è la dimensione; sotto, i temi che ne hanno una sola.

**Risultato.**

- **10 temi su 22 in entrambi** (nucleo compreso): l'agenda di fondo è comune, in particolare le vicende del 2025-2026 (Minneapolis, Los Angeles, Chicago, campus pro-Palestina) e i filoni istituzionali (tribunali, partiti e Congresso).
- **Peso diverso**: il NYT dà più spazio a *Trump administration* (14% dei nodi contro 5%), il Guardian a *Latin America, El Salvador* (12% contro 3%) e a Minneapolis (6% contro 3%): uno sguardo più interno e istituzionale contro uno più internazionale.
- **11 temi solo nel NYT**, quasi tutti locali o di politica interna: New York (città e Stato), Newark, confine e asilo, studenti e visti, shutdown, lavoro dei migranti, anni di Obama. In parte è un effetto di dimensione (il NYT ha più keyword e più articoli prima del 2025: le vicende piccole raggiungono il peso 5), in parte di agenda (New York è la città del giornale). **Solo nel Guardian**: cultura e televisione (tag annidati).
- Limite: l'allineamento dei temi è un giudizio (es. *Shootings and deaths during operations* NYT con *Texas and gun crime* Guardian è un accostamento largo). Va dichiarato nel report; la tabella di corrispondenza keyword NYT ↔ tag Guardian (ancora da fare) permetterebbe un confronto più preciso.

## 6. Comunità degli autori (NYT)

Si legge la partizione rappresentativa della §4 sulla rete degli autori: backbone del disparity filter con α = 0,05, Leiden con modularità, γ = 2. Solo comunità con almeno 5 autori. Gli autori inclusi sono 358 (almeno 2 articoli ICE), su tutto il 2003-2026: dal 2026-10-05 le firme del 2025+ si leggono dal testo `byline.original` (§3.1). Prima di questa correzione la §6 copriva solo il 2003-2024 con 117 autori e 7 comunità.

Cinque domande, un grafico ciascuna:

1. **Chi c'è in ogni comunità** (§6.1): autori principali e keyword che li accomunano.
2. **Quando hanno scritto** (§6.2): le comunità sono gruppi tematici o generazioni di giornalisti?
3. **Controllo del tempo** (§6.3): se togliamo dai profili le persone che identificano un'amministrazione (presidenti, segretari del DHS...), le comunità restano?
4. **Robustezza** (§6.4): le comunità restano con un backbone più severo (α = 0,01)?
5. **Temi coperti** (§6.5): quali comunità di keyword della §5 coprono le comunità di autori.

**Etichette**: come in §5, scritte a mano e legate a un autore "firma" (`AUTHOR_COMMUNITY_LABELS`); la cella si ferma se una firma manca, se due firme stanno nella stessa comunità o se una comunità leggibile resta senza etichetta.

In [ ]:
AUTHOR_KEY = ("nyt", "autori, backbone α=0,05")
AUTHOR_CONTROL_KEY = ("nyt", "autori, controllo α=0,01")
AUTHOR_PERIODS = PERIOD_LABELS

# Etichette scritte a mano leggendo autori e keyword caratterizzanti; chiave = un autore "firma" della comunità.
AUTHOR_COMMUNITY_LABELS = {
    "Julia Preston": "Immigration and deportation, 2003-2016 (Preston, Semple)",
    "Miriam Jordan": "Border, asylum and family separation",
    "Luis Ferré-Sadurní": "New York and New Jersey (Delaney Hall)",
    "Michael Gold": "Congress, parties and opinion columns",
    "Charlie Savage": "Los Angeles and California",
    "Allison McCann": "Shootings and deaths during operations",
    "Alan Feuer": "Courts and El Salvador deportations",
    "Jack Healy": "Campus protests and foreign students",
    "Nicholas Nehamas": "Contracts, DOGE and surveillance",
    "Julie Bosman": "Chicago, Portland and the National Guard",
    "Karoun Demirjian": "Shutdown and airport security",
    "Ernesto Londoño": "Minneapolis (2026)",
    "Frances Robles": "Foreign policy (Venezuela, Latin America)",
    "Will Jarvis": "News roundups (podcasts, newsletters)",
    "Ben Sisario": "Culture and sports",
    "Andrew Ross Sorkin": "Business (DealBook)",
}

signatures = author_rows[author_rows["author"].isin(authors)].merge(
    pd.read_parquet(PAPERS["nyt"]["articles"], columns=["id", "pub_date"]), on="id")
signatures["period"] = signatures["pub_date"].dt.year.map(period_of)


def labelled_communities(partition: pd.Series, labels: dict[str, str]) -> dict[int, str]:
    """community -> etichetta, dalla firma; errore se una firma manca, è doppia o una comunità leggibile resta senza etichetta."""
    out = {}
    for signature, label in labels.items():
        if signature not in partition.index:
            raise KeyError(f"autore firma '{signature}' non è nella rete")
        community = int(partition[signature])
        if community in out:
            raise ValueError(f"'{signature}' e un'altra firma sono nella stessa comunità {community}")
        out[community] = label
    sizes = partition.value_counts()
    missing = set(sizes[sizes >= MIN_COMMUNITY_SIZE].index) - set(out)
    if missing:
        raise ValueError(f"comunità leggibili senza etichetta: {sorted(missing)}")
    return out


def centroid_keywords(partition: pd.Series, profiles: pd.DataFrame, n: int = 3) -> dict[int, list[str]]:
    """Keyword con il peso più alto nel profilo medio (TF-IDF normalizzato) degli autori della comunità:
    sono le keyword che determinano la similarità coseno tra gli autori del gruppo."""
    weighted = tfidf(profiles)
    unit = weighted.div(np.sqrt((weighted ** 2).sum(axis=1)), axis=0)
    return {c: unit.loc[members].mean().nlargest(n).index.tolist()
            for c, members in partition.groupby(partition).groups.items()}


def author_summary(partition: pd.Series, profiles: pd.DataFrame, labels: dict[str, str]) -> pd.DataFrame:
    """Una riga per comunità leggibile di autori."""
    names = labelled_communities(partition, labels)
    keywords = centroid_keywords(partition, profiles)
    rows = []
    for community, label in names.items():
        members = partition[partition == community].index
        top = author_articles[members].sort_values(ascending=False)
        rows.append({
            "community": community, "etichetta": label, "autori": len(members),
            "articoli": signatures.loc[signatures["author"].isin(members), "id"].nunique(),
            "autori_principali": " · ".join(top.index[:3]),
            "keyword_caratterizzanti": " · ".join(shorten(k) for k in keywords[community]),
            "anno_mediano": int(signatures.loc[signatures["author"].isin(members), "pub_date"].dt.year.median()),
        })
    return pd.DataFrame(rows).sort_values("autori", ascending=False).reset_index(drop=True)


AUTHOR_SUMMARY = author_summary(partitions[AUTHOR_KEY], profiles, AUTHOR_COMMUNITY_LABELS)
n_authors = len(partitions[AUTHOR_KEY])
log.info("Autori: %d comunità leggibili, %d autori su %d", len(AUTHOR_SUMMARY), AUTHOR_SUMMARY["autori"].sum(), n_authors)

**Come leggere il log.** Quante comunità hanno almeno 5 autori e quanti dei 117 autori coprono.

### 6.1 Profilo delle comunità di autori

Due pannelli, una riga per comunità:

- **autori**: quanti, e i 3 con più articoli ICE;
- **articoli firmati**: quanti articoli ICE distinti ha firmato almeno un autore del gruppo, e le **keyword caratterizzanti**. Sono le keyword con il peso più alto nel profilo medio del gruppo (profili TF-IDF normalizzati come per il coseno della §3): quelle che rendono simili gli autori del gruppo, non le più frequenti.

Tra parentesi dopo l'etichetta, l'anno mediano delle firme del gruppo.

In [ ]:
def plot_author_profile(summary: pd.DataFrame, out_path: Path) -> None:
    """Due pannelli con le comunità sulle righe: autori (con i 3 più prolifici) e articoli firmati (con le keyword caratterizzanti)."""
    y = np.arange(len(summary))[::-1]
    fig, (ax_auth, ax_art) = plt.subplots(1, 2, figsize=(13, 0.42 * len(summary) + 1.4), sharey=True,
                                          gridspec_kw={"wspace": 0.05})
    for ax, col, text_col, xlabel in [
        (ax_auth, "autori", "autori_principali", "authors in the community  (text: the 3 with most ICE articles)"),
        (ax_art, "articoli", "keyword_caratterizzanti", "signed ICE articles  (text: characteristic keywords)"),
    ]:
        ax.barh(y, summary[col], color=PAPER_COLORS["nyt"], height=0.65)
        for yi, value, text in zip(y, summary[col], summary[text_col]):
            ax.text(value + summary[col].max() * 0.02, yi, f"{value}   {text}", va="center", fontsize=7, color=INK_SECONDARY)
        ax.set_xlim(0, summary[col].max() * (3.4 if col == "autori" else 2.9))
        ax.set_xlabel(xlabel)
        ax.grid(axis="x", color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)
        ax.tick_params(axis="y", length=0)
    ax_auth.set_yticks(y)
    ax_auth.set_yticklabels([f"{label}  ({year})" for label, year in zip(summary["etichetta"], summary["anno_mediano"])])
    fig.suptitle(f"NYT — author communities (backbone α = {BACKBONE_ALPHA}, γ = {CHOSEN_RESOLUTION[AUTHOR_KEY]:g}): {len(summary)} communities with at least "
                 f"{MIN_COMMUNITY_SIZE} authors, {summary['autori'].sum()} of {n_authors} authors  (median signature year in brackets)",
                 x=0.02, ha="left", fontsize=11, color=INK)
    fig.savefig(out_path)
    plt.show()


plot_author_profile(AUTHOR_SUMMARY, FIGURES_DIR / "community_profile_authors.png")

**Come leggere il grafico.** Ogni riga è un gruppo di giornalisti che scrivono degli stessi temi su ICE; a sinistra chi sono, a destra quanto hanno scritto e cosa li accomuna.

**Risultato.** 16 comunità leggibili coprono 355 autori su 358. Quasi tutte corrispondono a **filoni o vicende**, spesso le stesse comunità di keyword della §5:

- due filoni degli anni precedenti: *Immigration and deportation, 2003-2016* (27 autori, 219 articoli: Preston, Bernstein, Semple) e *Border, asylum and family separation* (36 autori, 333 articoli: Jordan, Kanno-Youngs, Dickerson), il gruppo più grande, a cavallo tra la prima amministrazione Trump e il 2025;
- le redazioni e le vicende del 2025-2026: *New York and New Jersey* (35 autori, 265 articoli: Ferré-Sadurní, Ana Ley, Robbins), *Los Angeles and California* (30), *Shootings and deaths during operations* (29), *Courts and El Salvador deportations* (22), *Campus protests and foreign students* (22), *Chicago, Portland and the National Guard* (20), *Minneapolis* (16), *Shutdown and airport security* (19);
- gruppi per tipo di giornalismo più che per tema: *Congress, parties and opinion columns* (34 autori: Michael Gold, gli editorialisti Jamelle Bouie e David French), *Contracts, DOGE and surveillance* (inchieste, 21), *Foreign policy* (15), *News roundups* (14: podcast e newsletter), *Culture and sports* (9) e *Business (DealBook)* (6).

**Hamed Aleaziz**, l'autore con più articoli ICE (173), è in un gruppo di 3 autori non leggibile: firma insieme a giornalisti di quasi tutti i gruppi e il suo profilo è generico (§3.2), quindi la modularità non lo assegna a nessun tema. È coerente con il suo ruolo di gatekeeper nel notebook 03 (betweenness più alta): è un ponte tra i gruppi, non il centro di uno.

### 6.2 Le comunità nel tempo

Per ogni comunità, quota delle sue firme (coppie autore-articolo) in ciascun periodo; l'ultima riga è il riferimento con tutti i 117 autori. In più, un numero: l'**AMI tra la partizione e il periodo prevalente** di ogni autore (quello in cui ha firmato più articoli ICE). 1 = le comunità coincidono con i periodi; 0 = nessun legame.

In [ ]:
def period_shares(partition: pd.Series, summary: pd.DataFrame) -> pd.DataFrame:
    """% delle firme di ogni comunità in ciascun periodo, più la riga di riferimento con tutti gli autori inclusi."""
    sig = signatures[signatures["author"].isin(partition.index)].assign(community=lambda d: d["author"].map(partition))
    table = pd.crosstab(sig["community"], sig["period"], normalize="index").reindex(columns=AUTHOR_PERIODS, fill_value=0) * 100
    table = table.loc[summary["community"]]
    table.index = summary["etichetta"]
    table.loc["all included authors"] = sig["period"].value_counts(normalize=True).reindex(AUTHOR_PERIODS, fill_value=0) * 100
    return table


def main_period_ami(partition: pd.Series) -> float:
    """AMI tra la partizione e il periodo in cui ogni autore ha firmato più articoli ICE (1 = le comunità sono i periodi)."""
    main_period = signatures.groupby("author")["period"].agg(lambda s: s.value_counts().index[0])
    return adjusted_mutual_info_score(partition, main_period[partition.index])


def plot_period_shares(table: pd.DataFrame, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 0.36 * len(table) + 1.4))
    ax.imshow(table.to_numpy(), cmap="Blues", vmin=0, vmax=100, aspect="auto")
    for (i, j), v in np.ndenumerate(table.to_numpy()):
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 60 else INK)
    n_by_period = signatures["period"].value_counts()
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels([f"{p.replace('presente', 'present')}\n({n_by_period.get(p, 0)} signatures)" for p in table.columns], fontsize=8)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(table)))
    ax.set_yticklabels(table.index)
    ax.axhline(len(table) - 1.5, color="white", linewidth=3)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, loc="left", fontsize=10, color=INK, pad=30)
    fig.savefig(out_path)
    plt.show()


AUTHOR_PERIOD_AMI = main_period_ami(partitions[AUTHOR_KEY])
plot_period_shares(period_shares(partitions[AUTHOR_KEY], AUTHOR_SUMMARY),
                   f"NYT — author community signatures by period (% of row)\n"
                   f"AMI between communities and each author's main period: {AUTHOR_PERIOD_AMI:.2f}",
                   FIGURES_DIR / "community_authors_periods.png")

**Come leggere il grafico.** Ogni riga dice in quali anni ha scritto il gruppo. Si confronta con l'ultima riga: una comunità che ha la stessa distribuzione di "all included authors" non è legata a un periodo; una tutta in una colonna sì.

**Risultato.**

- Il 70% delle firme è del 2025-presente: la rete autori ora è dominata dal secondo mandato Trump, come il corpus (1.224 articoli su 2.093).
- **Due comunità appartengono a periodi precedenti**: *Immigration and deportation, 2003-2016* (80% delle firme nel 2003-2016) e *Border, asylum and family separation* (52% nel 2017-2020, 15% nel 2021-2024, 32% nel 2025+), l'unica che attraversa tre periodi. *Shutdown and airport security* ha un terzo delle firme nel 2017-2020 (lo shutdown del 2018-2019) e il resto nel 2025+; *Culture and sports* è distribuita su tutti i periodi.
- Le altre dodici sono quasi tutte nel 2025+ (77-98%): si distinguono quindi per **tema e redazione dentro lo stesso periodo**, non per epoca.
- L'AMI tra comunità e periodo prevalente dell'autore è basso (0,12): nel complesso la partizione non riproduce i periodi.

### 6.3 Controllo del tempo: profili senza le keyword di amministrazione

La similarità tra autori si calcola sulle keyword dei loro articoli. Alcune keyword identificano un'amministrazione più che un tema ("obama, barack", "trump, donald j", "napolitano, janet"): due giornalisti dello stesso periodo le condividono anche se scrivono di cose diverse, e questo potrebbe creare comunità "di epoca".

Controllo: si tolgono dai profili tutte le keyword di **presidenti, vicepresidenti, segretari del DHS, attorney general, direttori dell'ICE** e dei due consiglieri più legati alla politica migratoria (Cuccinelli, Stephen Miller), elencate in `ADMINISTRATION_KEYWORDS` dopo averle verificate sulle keyword `persons` degli articoli 2003-2024. Poi si ripete tutta la pipeline delle §3-4 con gli stessi parametri (TF-IDF, coseno, backbone α = 0,05, Leiden γ = 1, 10 seed) e si confronta:

- **AMI con la partizione originale**: le comunità restano?
- **AMI con il periodo prevalente**: scende? Se le keyword di amministrazione legavano le comunità al tempo, toglierle dovrebbe abbassarlo.
- **Dove finiscono** gli autori di ogni comunità originale (mappa di colore come in §5.3).

In [ ]:
# Persone che identificano un'amministrazione più che un tema: presidenti, vicepresidenti, segretari del DHS,
# attorney general, direttori dell'ICE (anche ad interim) e i due consiglieri più legati alla politica migratoria.
# Elenco verificato sulle keyword `persons` degli articoli ICE 2003-2026.
ADMINISTRATION_KEYWORDS = {
    # presidenti e vicepresidenti
    "bush, george w", "obama, barack", "trump, donald j", "biden, joseph r jr",
    "pence, mike", "harris, kamala d", "vance, j d",
    # segretari del DHS
    "napolitano, janet", "johnson, jeh c", "kelly, john f (1950- )", "nielsen, kirstjen",
    "mcaleenan, kevin k", "wolf, chad f", "mayorkas, alejandro", "noem, kristi",
    "mullin, markwayne",  # segretario dal 2026
    # attorney general
    "sessions, jefferson b iii", "barr, william p", "bondi, pamela j",
    # direttori dell'ICE
    "myers, julie", "morton, john", "morton, john t", "saldana, sarah r", "homan, thomas d",
    "vitiello, ronald d", "albence, matthew t", "lyons, todd m", "schroyer, lance",
    # consiglieri: Cuccinelli (USCIS, vice segretario DHS ad interim), Stephen Miller (Casa Bianca)
    "cuccinelli, kenneth t ii", "miller, stephen (1985- )",
}


def detect_author_communities(rows: pd.DataFrame) -> tuple[pd.Series, nx.Graph]:
    """Stessa pipeline delle §3-4 (profili TF-IDF, coseno, backbone α, Leiden γ = 1, partizione rappresentativa)."""
    backbone = largest_component(disparity_filter(similarity_graph(tfidf(author_profiles(rows, authors))), BACKBONE_ALPHA))
    g, nodes = to_igraph(backbone, "weight")
    runs = [leiden(g, "modularità", CHOSEN_RESOLUTION[AUTHOR_KEY], seed) for seed in range(N_SEEDS)]
    log.info("Senza keyword di amministrazione: %d autori, %d archi, AMI tra seed %.2f",
             backbone.number_of_nodes(), backbone.number_of_edges(), mean_pairwise_ami([np.array(r.membership) for r in runs]))
    return representative_partition(runs, nodes), backbone


in_profiles = author_rows[author_rows["author"].isin(authors)]
removed = in_profiles["keywords"].apply(lambda kws: sum(k in ADMINISTRATION_KEYWORDS for k in kws)).sum()
log.info("Keyword di amministrazione nei profili: %d occorrenze su %d (%.1f%%), %d keyword distinte",
         removed, in_profiles["keywords"].apply(len).sum(), removed / in_profiles["keywords"].apply(len).sum() * 100,
         len(ADMINISTRATION_KEYWORDS & set(profiles.columns)))

rows_no_admin = author_rows.assign(keywords=author_rows["keywords"].apply(lambda kws: [k for k in kws if k not in ADMINISTRATION_KEYWORDS]))
partition_no_admin, backbone_no_admin = detect_author_communities(rows_no_admin)
partitions[("nyt", "autori, senza amministrazioni")] = partition_no_admin

shared = partitions[AUTHOR_KEY].index.intersection(partition_no_admin.index)
NO_ADMIN_AMI = adjusted_mutual_info_score(partitions[AUTHOR_KEY][shared], partition_no_admin[shared])
NO_ADMIN_PERIOD_AMI = main_period_ami(partition_no_admin)
log.info("AMI con la partizione originale: %.2f; AMI con il periodo prevalente: %.2f (originale %.2f)",
         NO_ADMIN_AMI, NO_ADMIN_PERIOD_AMI, AUTHOR_PERIOD_AMI)


def plot_partition_overlap(rows: pd.Series, cols: pd.Series, row_labels: dict[int, str], title: str, out_path: Path) -> pd.Series:
    """Righe = comunità leggibili di `rows` (con etichetta), colonne = comunità di `cols`: % dei nodi della riga in quella colonna."""
    shared = rows.index.intersection(cols.index)
    sub = pd.DataFrame({"row": rows[shared], "col": cols[shared]})
    sub = sub[sub["row"].isin(row_labels)]
    table = pd.crosstab(sub["row"], sub["col"], normalize="index") * 100
    table = table.loc[[c for c in AUTHOR_SUMMARY["community"] if c in table.index]]
    order = sorted(table.columns, key=lambda c: (int(np.argmax(table[c].to_numpy() == table[c].max())), -table[c].max()))
    table = table[order]
    table.index = [row_labels[c] for c in table.index]

    fig, ax = plt.subplots(figsize=(0.4 * table.shape[1] + 4.5, 0.4 * table.shape[0] + 1.4))
    ax.imshow(table.to_numpy(), cmap="Blues", vmin=0, vmax=100, aspect="auto")
    for (i, j), v in np.ndenumerate(table.to_numpy()):
        if v >= 10:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 60 else INK)
    ax.set_yticks(range(len(table)))
    ax.set_yticklabels(table.index)
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels(table.columns, fontsize=7)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
    fig.savefig(out_path)
    plt.show()
    return table.max(axis=1).rename("quota_max_pct")


author_labels = labelled_communities(partitions[AUTHOR_KEY], AUTHOR_COMMUNITY_LABELS)
NO_ADMIN_AGREEMENT = plot_partition_overlap(
    partitions[AUTHOR_KEY], partition_no_admin, author_labels,
    f"NYT authors — where each community's authors end up without administration keywords\n"
    f"AMI with the original partition {NO_ADMIN_AMI:.2f}; AMI with the authors' main period {AUTHOR_PERIOD_AMI:.2f} → {NO_ADMIN_PERIOD_AMI:.2f}\n"
    f"(columns = communities of the new partition; colour = % of the row's authors, number shown if ≥ 10%)",
    FIGURES_DIR / "community_authors_no_admin.png",
)

**Come leggere il grafico.** Righe = comunità originali, colonne = comunità ottenute senza le keyword di amministrazione (solo numeri). Il numero è la % degli autori della riga che finisce in quella colonna. Una riga con una sola cella scura = la comunità esiste anche senza i nomi delle amministrazioni.

**Risultato.**

- Le keyword tolte sono l'8,2% delle occorrenze nei profili (30 keyword distinte, 1.956 occorrenze; nel 2025+ entrano Noem, Mullin, Homan, Lyons, Bondi, Vance, Stephen Miller): una parte piccola ma molto condivisa.
- **Le comunità restano**: AMI 0,84 con la partizione originale; 14 comunità su 16 restano per almeno il 75% nella stessa nuova comunità, sette al 94-100%.
- **Si dividono in parte** *Shutdown and airport security* (47% + 47%) e *Congress, parties and opinion columns* (59% + 38%): sono i gruppi di politica nazionale, tenuti insieme anche dai nomi dei vertici dell'amministrazione. Più leggermente *Courts and El Salvador deportations* (77% + 23%) e *Chicago, Portland and the National Guard* (75% + 20%).
- **Il legame con il tempo non cambia** (AMI con il periodo 0,12 → 0,13). La comunità del 2003-2016 non dipende dai nomi delle amministrazioni ma dal **vocabolario della tassonomia NYT**, che cambia nel tempo: "illegal immigrants" è usata solo nel 2008-2013 e "immigration and refugees" solo nel 2008-2009, poi sostituite da "illegal immigration" e "immigration and emigration". Questo effetto non si può togliere senza una mappatura manuale dei tag tra epoche. Va dichiarato nel report: la comunità *Immigration and deportation, 2003-2016* è insieme un filone e una generazione.
- **Conclusione**: le comunità di autori non sono un artefatto dei nomi delle amministrazioni. Si tiene la partizione originale; quella di controllo resta in `partitions` come `("nyt", "autori, senza amministrazioni")`.

### 6.4 Robustezza: backbone α = 0,05 contro α = 0,01

Stesso controllo della §5.3, sulla rete degli autori: la rete di controllo tiene solo gli archi che spiccano con α = 0,01 (265 archi invece di 694, §3.3). Se gli autori di una comunità restano insieme anche così, la comunità non dipende dalla soglia del disparity filter.

In [ ]:
shared = partitions[AUTHOR_KEY].index.intersection(partitions[AUTHOR_CONTROL_KEY].index)
ALPHA_AMI = adjusted_mutual_info_score(partitions[AUTHOR_KEY][shared], partitions[AUTHOR_CONTROL_KEY][shared])
ALPHA_AGREEMENT = plot_partition_overlap(
    partitions[AUTHOR_KEY], partitions[AUTHOR_CONTROL_KEY], author_labels,
    f"NYT authors — where each community's authors end up with the stricter backbone (α = 0.01)\n"
    f"AMI {ALPHA_AMI:.2f}  (columns = control network communities; colour = % of the row's authors, number shown if ≥ 10%)",
    FIGURES_DIR / "community_authors_vs_control.png",
)
log.info("α 0,05 contro 0,01: AMI %.2f; comunità con ≥ 50%% degli autori in un'unica comunità di controllo: %d su %d",
         ALPHA_AMI, (ALPHA_AGREEMENT >= 50).sum(), len(ALPHA_AGREEMENT))

**Come leggere il grafico.** Come in §6.3: righe = comunità con α = 0,05, colonne = comunità con α = 0,01, numero = % degli autori della riga.

**Risultato.** AMI 0,68, più alto che per le reti di keyword (0,38-0,39); **14 comunità su 16** hanno almeno metà degli autori in un'unica comunità di controllo. Le più stabili: *New York and New Jersey* (97%), *Immigration and deportation, 2003-2016* (96%), *Foreign policy* (93%), *Minneapolis* (81%), *Culture and sports* e *Business* (100%, ma piccole). Con il backbone più severo si dividono *Courts and El Salvador deportations* (al massimo 41% insieme) e *Shutdown and airport security* (42% + 32%), e in parte *Border, asylum and family separation* (56% + 31%) e *Congress, parties and opinion columns* (62% + 32%): di nuovo i gruppi di politica nazionale, i più trasversali.

### 6.5 Temi coperti dalle comunità di autori

Collega le due reti: per ogni comunità di autori e ogni comunità di keyword NYT della §5 (nucleo escluso), la quota degli articoli firmati dal gruppo che contiene almeno una keyword del tema. Il colore confronta questa quota con la media di tutti gli articoli firmati: rapporto in scala log2, rosso = il gruppo copre il tema più della media, blu = meno. La scala è tagliata a ±2 (4 volte sopra o sotto la media) perché i gruppi piccoli danno valori estremi.

Sono esclusi i temi presenti in meno del 2% degli articoli firmati (`MIN_THEME_PRESENCE`): sono le vicende del 2025-2026, che non hanno articoli con firma.

In [ ]:
import textwrap
from matplotlib.colors import LinearSegmentedColormap

# divergente: blu (sotto la media) - grigio neutro - rosso (sopra la media)
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#1c5cab", "#86b6ef", "#f0efec", "#ef9a9a", "#c62f2f"])
MIN_THEME_PRESENCE = 2.0  # % minima degli articoli firmati: sotto, il tema non è coperto dagli articoli con firma
LIFT_LIMIT = 2.0  # scala colori tagliata a ±2 (4 volte sopra/sotto la media): i gruppi piccoli darebbero valori estremi


def theme_coverage(summary: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Per comunità di autori × comunità di keyword NYT (§5): % degli articoli firmati dal gruppo con almeno una
    keyword del tema, e rapporto (log2) con la stessa quota su tutti gli articoli firmati dagli autori inclusi."""
    keyword_groups = readable_communities(MAIN_KEY["nyt"])
    keyword_labels = community_labels("nyt")
    core = int(partitions[MAIN_KEY["nyt"]][CORE_KEYWORD["nyt"]])
    themes = [c for c in SUMMARIES["nyt"]["community"] if c != core]

    article_keywords = signatures.drop_duplicates("id").set_index("id")["keywords"].apply(set)
    article_group = signatures.assign(community=signatures["author"].map(partitions[AUTHOR_KEY])).groupby("id")["community"].agg(set)
    presence = {}
    for theme in themes:
        members = set(keyword_groups[theme])
        has = article_keywords.apply(lambda kws: not kws.isdisjoint(members))
        row = {"tutti": has.mean() * 100}
        for community, label in zip(summary["community"], summary["etichetta"]):
            ids = article_group[article_group.apply(lambda s: community in s)].index
            row[label] = has[ids].mean() * 100
        presence[keyword_labels[theme]] = row
    presence = pd.DataFrame(presence).T
    presence = presence[presence["tutti"] >= MIN_THEME_PRESENCE]
    lift = np.log2((presence.drop(columns="tutti") + 0.5).div(presence["tutti"] + 0.5, axis=0))
    return presence, lift


def plot_theme_coverage(presence: pd.DataFrame, lift: pd.DataFrame, summary: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(0.95 * lift.shape[1] + 5, 0.34 * lift.shape[0] + 2.2))
    vmax = LIFT_LIMIT
    im = ax.imshow(lift.to_numpy().clip(-vmax, vmax), cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect="auto")
    for (i, j), v in np.ndenumerate(presence.drop(columns="tutti").to_numpy()):
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                color="white" if abs(lift.iloc[i, j]) > vmax * 0.65 else INK)
    ax.set_xticks(range(lift.shape[1]))
    n_articles = dict(zip(summary["etichetta"], summary["articoli"]))
    ax.set_xticklabels([f"{textwrap.fill(c, 16)}\n({n_articles[c]} articles)" for c in lift.columns], fontsize=7)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(lift)))
    ax.set_yticklabels([f"{t}  ({v:.0f}%)" for t, v in presence["tutti"].items()])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("log2(group share / share of all signed articles), clipped at ±2", fontsize=7)
    cbar.outline.set_visible(False)
    ax.set_title("NYT — which themes (keyword communities, §5) the author communities cover\n"
                 "number = % of the group's articles with at least one theme keyword; colour = above (red) or below (blue) average; "
                 "average in brackets", loc="left", fontsize=9, color=INK, pad=52)
    fig.savefig(out_path)
    plt.show()


THEME_PRESENCE, THEME_LIFT = theme_coverage(AUTHOR_SUMMARY)
excluded = sorted(set(community_labels("nyt").values()) - set(THEME_PRESENCE.index) - {SUMMARIES["nyt"].loc[SUMMARIES["nyt"]["nucleo"], "etichetta"].iloc[0]})
log.info("Temi esclusi (presenti in meno del %.0f%% degli articoli firmati): %s", MIN_THEME_PRESENCE, ", ".join(excluded))
plot_theme_coverage(THEME_PRESENCE, THEME_LIFT, AUTHOR_SUMMARY, FIGURES_DIR / "community_authors_themes.png")

**Come leggere il grafico.** Righe = temi della §5 con, tra parentesi, la quota media sugli articoli firmati; colonne = comunità di autori con il numero di articoli. Il numero nella cella è la % degli articoli del gruppo con il tema; il colore dice se è sopra o sotto la media. Si legge per colonna (di cosa scrive quel gruppo) o per riga (chi copre quel tema). Le colonne *Culture and sports* (22 articoli) e *Business* (16) sono piccole: ogni articolo sposta la quota di 5-6 punti, non vanno interpretate.

**Risultato.** Le comunità di autori corrispondono ai temi della §5, e questo conferma le etichette di due reti costruite in modo indipendente (una dalle keyword condivise negli articoli, l'altra dalla somiglianza dei profili degli autori):

- *Border, asylum and family separation* → **Border, asylum and family separation** (54% contro 25%);
- *New York and New Jersey* → **New York City: Adams and Rikers** (43% contro 11%), **Delaney Hall** (27% contro 9%), **New York State** (22% contro 9%);
- *Congress, parties and opinion columns* → **Parties and elections** (60% contro 11%), **Shutdown** (36% contro 12%);
- *Los Angeles and California* → **Protests and National Guard** (58% contro 20%);
- *Shootings and deaths during operations* → **Shootings** (37% contro 10%) e Minneapolis (30%);
- *Immigration and deportation, 2003-2016* → **Obama years** (49% contro 8%);
- *Courts and El Salvador deportations* → **Courts** (53% contro 21%) e **El Salvador** (20% contro 7%);
- *Campus protests and foreign students* → **Foreign students and visas** (44% contro 7%), **Pro-Palestinian campus protests** (25% contro 3%);
- *Chicago, Portland and the National Guard* → **Chicago** (39% contro 13%), **Military and Guantánamo** (28% contro 5%);
- *Minneapolis* → **Minneapolis** (54% contro 14%); *Foreign policy* → **International relations** (60% contro 11%);
- *Contracts, DOGE and surveillance* copre più della media **Delaney Hall** (33%): le inchieste sui contratti dei centri di detenzione.

Con il 2025+ incluso nessun tema resta escluso per mancanza di firme.

## 7. Comunità per periodo: non svolta

Era prevista una community detection separata su ciascuno dei quattro periodi. **Decisione (2026-10-05): non si fa**, per due motivi.

1. **L'evoluzione dei temi nel tempo è già coperta**: §5.2 (presenza delle comunità di keyword per periodo), notebook 02 (grafi per periodo, bubble chart, keyword distintive per anno), notebook 04 e 07 (topic BERTopic per periodo, NYT e Guardian). La §7 aggiungerebbe solo come cambiano i *legami* tra temi dentro ogni periodo, un'informazione marginale rispetto a queste.
2. **Le reti di alcuni periodi sono troppo piccole** per comunità affidabili: il 2021-2024 ha 96 articoli NYT e 90 Guardian, il Guardian 2003-2016 108. Le partizioni sarebbero instabili e difficili da difendere nel report.

Le reti per periodo restano disponibili (GEXF dei notebook 02 e 08) se servissero in seguito.

## 8. Export e note per il report

Si salvano le partizioni rappresentative come attributi dei nodi, per Gephi e per i passi successivi (es. scelta delle keyword per Telegram):

- un **GEXF per rete** in `data/processed/{nyt,guardian}/communities/`: rete diluita e di controllo delle keyword (NYT e Guardian), rete autori (backbone α = 0,05, controllo α = 0,01, senza keyword di amministrazione). Sono le reti su cui è stato eseguito Leiden, cioè la **componente maggiore**. Attributi aggiunti ai nodi: `community` (numero, 0 = la più grande), `community_size`, `community_label` (etichetta inglese, vuota per le comunità sotto i 5 nodi e per le reti di controllo, che non sono state etichettate), `ice_articles` per gli autori. Gli archi tengono `weight` e, per le keyword, `assoc`;
- una **tabella unica** `data/processed/community_membership.csv`: una riga per nodo e rete, con le stesse colonne.

In Gephi: *Appearance → Nodes → Partition → community_label* (o `community`) colora i nodi per comunità.

In [ ]:
EXPORTS = {
    # chiave in `partitions` -> (giornale, nome del file, etichette {community: label} o None)
    ("nyt", "keyword, diluita"): ("nyt", "keywords_pruned", community_labels("nyt")),
    ("nyt", "keyword, controllo"): ("nyt", "keywords_control", None),
    ("guardian", "keyword, diluita"): ("guardian", "keywords_pruned", community_labels("guardian")),
    ("guardian", "keyword, controllo"): ("guardian", "keywords_control", None),
    AUTHOR_KEY: ("nyt", "authors_backbone", author_labels),
    AUTHOR_CONTROL_KEY: ("nyt", "authors_control", None),
    ("nyt", "autori, senza amministrazioni"): ("nyt", "authors_no_admin", None),
}
NETWORK_GRAPHS = {**{key: graph for key, (graph, _) in NETWORKS.items()},
                  ("nyt", "autori, senza amministrazioni"): backbone_no_admin}


def annotate_communities(graph: nx.Graph, partition: pd.Series, labels: dict[int, str] | None) -> nx.Graph:
    """Copia del grafo con gli attributi di comunità sui nodi (GEXF non accetta None: stringa vuota se manca l'etichetta)."""
    out = graph.copy()
    sizes = partition.value_counts()
    for node, community in partition.items():
        data = out.nodes[node]
        data["community"] = int(community)
        data["community_size"] = int(sizes[community])
        data["community_label"] = (labels or {}).get(int(community), "")
        if node in author_articles.index:
            data["ice_articles"] = int(author_articles[node])
    return out


rows = []
for key, (paper, name, labels) in EXPORTS.items():
    out_dir = PAPERS[paper]["dir"] / "communities"
    out_dir.mkdir(exist_ok=True)
    graph = annotate_communities(NETWORK_GRAPHS[key], partitions[key], labels)
    path = out_dir / f"{name}.gexf"
    nx.write_gexf(graph, path)
    for node, data in graph.nodes(data=True):
        rows.append({"paper": paper, "network": name, "node": node, "node_label": data.get("label", node),
                     "community": data["community"], "community_size": data["community_size"],
                     "community_label": data["community_label"]})
    log.info("Scritto %s (%d nodi, %d comunità, %d con etichetta)", path.relative_to(BASE_DIR),
             graph.number_of_nodes(), partitions[key].nunique(), len(labels or {}))

membership = pd.DataFrame(rows)
OUT_MEMBERSHIP = BASE_DIR / "data" / "processed" / "community_membership.csv"
membership.to_csv(OUT_MEMBERSHIP, index=False)
log.info("Scritto %s (%d righe)", OUT_MEMBERSHIP.relative_to(BASE_DIR), len(membership))

**Come leggere il log.** Una riga per file scritto: nodi, comunità totali, comunità con etichetta (solo per le reti principali).

### 8.1 Note per il report

**Scelte metodologiche** (dettagli e motivazioni nelle sezioni indicate):

- Pesi delle reti di keyword normalizzati con **association strength** (osservate / attese, §2); rete principale = archi con peso ≥ 5 (notebook 02/08), rete di controllo = keyword in ≥ 5 articoli con tutti gli archi (filtro VOSviewer, §2.1).
- Rete autori: autori con ≥ 2 articoli ICE, profili **TF-IDF**, similarità **coseno**, backbone con il **disparity filter** α = 0,05 (§3).
- **Leiden con modularità**, γ = 3 per le keyword e γ = 2 per gli autori, 10 seed, partizione rappresentativa = AMI medio più alto; tutte le reti superano 20 reti casuali con gli stessi gradi (z 16-120, §4).

**Limiti da dichiarare:**

1. **Soglie scelte a mano**: peso ≥ 5, keyword ≥ 5 articoli, α = 0,05, γ. Sono state controllate con reti alternative (§5.3, §6.4) e con la stabilità tra seed (§4.2), ma restano scelte.
2. **Doppia correzione per la frequenza**: l'association strength corregge i pesi e la modularità corregge di nuovo nel modello nullo (§4.1). Scelta empirica, motivata dal confronto con CPM e con i conteggi grezzi.
3. **Modularità bassa** (0,08-0,18 per le keyword): la struttura è significativa ma i temi si sovrappongono; le comunità hanno confini sfumati (§4.3).
4. **Solo le comunità-vicenda sono robuste**: le comunità trasversali (nucleo, amministrazione Trump, tribunali, partiti) si sciolgono nella rete di controllo (§5.3) e vanno citate con cautela.
5. **Etichette e allineamento NYT-Guardian fatti a mano** (§5, §5.4, §6): sono interpretazioni, da leggere insieme alle keyword. Il confronto tra giornali è sui temi, non nodo per nodo (tassonomie diverse).
6. **Firme del 2025+ lette dal testo**: dal 2025 il NYT non compila più la lista strutturata degli autori; gli autori si leggono dalla firma testuale `byline.original` (98% degli articoli 2025+), con regole per firme istituzionali e fotografi e due alias (§3.1). Un errore di questa lettura non è escluso, ma il controllo dei nomi doppi non ha trovato altri casi.
7. **Il vocabolario della tassonomia NYT cambia nel tempo** ("illegal immigrants" solo 2008-2013, poi "illegal immigration"): la comunità di autori del 2003-2016 è insieme un filone e una generazione (§6.3).
8. **Dati NYT puliti a monte**: 28 tag tecnici del sistema editoriale (`internal-*`, `audio-*`, `vis-*`, `your-feed-*`...) esclusi e keyword duplicate tolte nel notebook 02 (§1).
9. **Comunità per periodo non calcolate** (§7).